# Temperature Profiles and Historical Comparison

This notebook processes and visualises temperature data from the Geoprecision thermistor chains and compares 2024/25 measurements with historical borehole records. It is divided into two main sections:

**Part 1 — Temperature profiles and heatmaps**
- Englacial temperature profiles combining GP chain and Tynitag measurements
- Temperature heatmap timeseries for each borehole
- Deployment timeline overview (Fig. 3)

**Part 2 — Historical comparison**
- Current temperatures at Sex Rouge compared with Fischer (2018) BH1 and BH2 profiles
- Current temperatures at Corvatsch compared with Haeberli et al. (2004) profiles
- Same-day-of-year ΔT analysis (Fig. 5)


---

## Part 1 — Temperature Profiles and Heatmaps

## Import required Libraries and Modules

In [ ]:
from config import ICETEMP_ROOT
import sys
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import cmcrameri.cm as cmc

mpl.rcParams.update({'font.family': 'Arial'})

# Add project root to Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.thermistor_processing import *
from src.thermistor_plotting import *
from src.gpr_plotting import *
from calibration.calibration_utils import compute_and_save_offsets
from calibration.thermistor_calibration import *
from calibration import thermistor_chains_icebath_references as chains


In [ ]:
# root dir
root_dir = ICETEMP_ROOT + "/"

# set main calibration data dir
cal_dir = os.path.join(root_dir, "thermistor_chains", "calibration_data") + "/"

# set chain calibration dir
A551FD_dir = cal_dir + "A551FD/raw/"
A551FE_dir = cal_dir + "A551FE/raw/"
A55200_dir = cal_dir + "A55200/raw/"
A55201_dir = cal_dir + "A55201/raw/"
A55202_dir = cal_dir + "A55202/raw/"
A55203_dir = cal_dir + "A55203/raw/"
A55204_dir = cal_dir + "A55204/raw/"
A55205_dir = cal_dir + "A55205/raw/"

# set maximum measurement depths
A551FE_depth = 45.0 # AH1 -> borehole depth is 50.6m
A55204_depth = 20.3 # AH2
A55205_depth = 18.3 # AH3 -> borehole depth is 58.3m
A551FD_depth = 29.0 # HL1
A55203_depth = 21.5 # HL2 
A55200_depth = 21.5 # HL3
A55201_depth = 38.3 # CH1
A55202_depth = 17.0 # CH2

# generate calibration data objects
A551FE_cal_data = ThermistorData(A551FE_dir + "A551FE_20250729123855.csv",",",A551FE_depth)   
A55204_cal_data = ThermistorData(A55204_dir + "A55204_20250729123756.csv",",",A55204_depth)   
A55205_cal_data = ThermistorData(A55205_dir + "A55205_20250729123810.csv",",",A55205_depth)   
A551FD_cal_data = ThermistorData(A551FD_dir + "A551FD_20250729123824.csv",",",A551FD_depth)
A55203_cal_data = ThermistorData(A55203_dir + "A55203_20250729123726.csv",",",A55203_depth)   
A55200_cal_data = ThermistorData(A55200_dir + "A55200_20250729123642.csv",",",A55200_depth)   
A55201_cal_data = ThermistorData(A55201_dir + "A55201_20250729123655.csv",",",A55201_depth)   
A55202_cal_data = ThermistorData(A55202_dir + "A55202_20250729123712.csv",",",A55202_depth)

In [ ]:
# set main icetemp data dir
gp_icetemp_dir = os.path.join(ICETEMP_ROOT, "thermistor_chains", "temperature_data") + "/"
TT_icetemp_dir = os.path.join(ICETEMP_ROOT, "NTC_tynitag", "temperature_data", "full_timeseries") + "/"

# set output dir for figures
output_dir = os.path.join(project_root, "products", "figures", "icetemp_results", "geoprecision") + "/"

# set chain data dir
A551FE_dir = gp_icetemp_dir + "2025/A551FE/raw/A551FE_20250916075641.csv" # AH1G
A55204_dir = gp_icetemp_dir + "2025/A55204/raw/A55204_20250916082002.csv" # AH2G
A55205_dir = gp_icetemp_dir + "2025/A55205/raw/A55205_20250916074654.csv" # AH3G
A551FD_dir = gp_icetemp_dir + "2025/A551FD/raw/A551FD_20250927153221.csv" # HS1G
A55203_dir = gp_icetemp_dir + "2025/A55203/raw/A55203_20250927151514.csv" # HS2G
A55200_dir = gp_icetemp_dir + "2025/A55200/raw/A55200_20250927151301.csv" # HS3G
A55201_dir = gp_icetemp_dir + "2025/A55201/raw/A55201_20251215143911.csv" # CJ1G
A55202_dir = gp_icetemp_dir + "2025/A55202/raw/A55202_20251215144715.csv" # CJ2G

# set tiny tag data dir
HS1TT_dir = TT_icetemp_dir + "HS1TT_20240808_20250927_spliced.csv" # HS1TT
HS2TT_dir = TT_icetemp_dir + "HS2TT_20240808_20250927_spliced.csv" # HS2TT
AH1TT_dir = TT_icetemp_dir + "AH1TT_20240821_20250916.csv" # AH1TT
AH2TT_dir = TT_icetemp_dir + "AH2TT_20240821_20250916_spliced.csv" # AH2TT
AH3TT_dir = TT_icetemp_dir + "AH3TT_20250806_20250916.csv" # AH3TT
CJ1TT_dir = TT_icetemp_dir + "CJ1TT_20240809_20250808_spliced.csv" # CJ1TT
CJ2TT_dir = TT_icetemp_dir + "CJ2TT_20240809_20250808_spliced.csv" # CJ2TT
CJ3TT_dir = TT_icetemp_dir + "/../2024_2025/new_naming/CJ3TT_20251215.csv" # CJ3TT
CJ4TT_dir = TT_icetemp_dir + "/../2024_2025/new_naming/CJ4TT_20251215.csv" # CJ4TT

## set path to current depth file

# Chessjen boreholes
depth_CJ1G = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_cj1g.csv")
depth_CJ2G = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_cj2g.csv")
depth_CJ1TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_cj1tt.csv")
depth_CJ2TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_cj2tt.csv")
depth_CJ3TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_cj3tt.csv")
depth_CJ4TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_cj4tt.csv")

# Alphubel boreholes
depth_AH1G = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_ah1g.csv")
depth_AH2G = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_ah2g.csv")
depth_AH3G = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_ah3g.csv")
depth_AH1TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_ah1tt.csv")
depth_AH2TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_ah2tt.csv")
depth_AH3TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_ah3tt.csv")

# Hohlaub boreholes (renamed to Hohsaas, HS)
depth_HS1G = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_hs1g.csv")
depth_HS2G = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_hs2g.csv")
depth_HS3G = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_hs3g.csv")
depth_HS1TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_hs1tt.csv")
depth_HS2TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_hs2tt.csv")

## generate chain plotting objects per borehole

# Chessjen boreholes
CJ1G_plotter = ThermistorDataPlotter(A55201_dir, ",")
CJ2G_plotter = ThermistorDataPlotter(A55202_dir, ",")
CJ1TT_plotter = ThermistorDataPlotter(CJ1TT_dir, ",")
CJ2TT_plotter = ThermistorDataPlotter(CJ2TT_dir, ",")

# Alphubel boreholes
AH1G_plotter = ThermistorDataPlotter(A551FE_dir, ",")
AH2G_plotter = ThermistorDataPlotter(A55204_dir, ",")
AH3G_plotter = ThermistorDataPlotter(A55205_dir, ",")
AH3TT_plotter = ThermistorDataPlotter(AH3TT_dir, ",")

# Hohsaas boreholes (HS)
HS1G_plotter = ThermistorDataPlotter(A551FD_dir, ",")
HS2G_plotter = ThermistorDataPlotter(A55203_dir, ",")
HS3G_plotter = ThermistorDataPlotter(A55200_dir, ",")
HS1TT_plotter = ThermistorDataPlotter(HS1TT_dir, ",")
HS2TT_plotter = ThermistorDataPlotter(HS2TT_dir, ",")

## generate a thermistor data object

# Chessjen boreholes
CJ1G = ThermistorData(A55201_dir, ",", depth_CJ1G)
CJ2G = ThermistorData(A55202_dir, ",", depth_CJ2G)
CJ1TT = ThermistorData(CJ1TT_dir, ",", depth_CJ1TT)
CJ2TT = ThermistorData(CJ2TT_dir, ",", depth_CJ2TT)
CJ3TT = ThermistorData(CJ3TT_dir, ",", depth_CJ3TT)
CJ4TT = ThermistorData(CJ4TT_dir, ",", depth_CJ4TT)

# Alphubel boreholes
AH1G = ThermistorData(A551FE_dir, ",", depth_AH1G)
AH2G = ThermistorData(A55204_dir, ",", depth_AH2G)
AH3G = ThermistorData(A55205_dir, ",", depth_AH3G)
AH1TT = ThermistorData(AH1TT_dir, ",", depth_AH1TT)
AH2TT = ThermistorData(AH2TT_dir, ",", depth_AH2TT)
AH3TT = ThermistorData(AH3TT_dir, ",", depth_AH3TT)

# Hohsaas boreholes (HS)
HS1G = ThermistorData(A551FD_dir, ",", depth_HS1G)
HS2G = ThermistorData(A55203_dir, ",", depth_HS2G)
HS3G = ThermistorData(A55200_dir, ",", depth_HS3G)
HS1TT = ThermistorData(HS1TT_dir, ",", depth_HS1TT)
HS2TT = ThermistorData(HS2TT_dir, ",", depth_HS2TT)


### Load data and calibration offsets

In [ ]:
# Load GP chain calibration offsets saved by notebook 2
_offsets_csv = os.path.join(project_root, 'data', 'calibration', 'corrected_chain_offsets.csv')
_df_offsets  = pd.read_csv(_offsets_csv, index_col='chain')

A551FE_offsets = _df_offsets.loc['A551FE'].dropna().to_dict()  # AH1G
A55204_offsets = _df_offsets.loc['A55204'].dropna().to_dict()  # AH2G
A55205_offsets = _df_offsets.loc['A55205'].dropna().to_dict()  # AH3G
A551FD_offsets = _df_offsets.loc['A551FD'].dropna().to_dict()  # HS1G
A55203_offsets = _df_offsets.loc['A55203'].dropna().to_dict()  # HS2G
A55200_offsets = _df_offsets.loc['A55200'].dropna().to_dict()  # HS3G
A55201_offsets = _df_offsets.loc['A55201'].dropna().to_dict()  # CJ1G
A55202_offsets = _df_offsets.loc['A55202'].dropna().to_dict()  # CJ2G


### Profiles

In [ ]:
## List containing file paths for all thermistor chain data per glacier

# Chessjengletscher
cj_file_paths = [A55201_dir, A55202_dir] 
CJ = ThermistorDataPlotter(cj_file_paths, delimiter=',')

# Alphubelgletscher
ah_file_paths = [A551FE_dir, A55204_dir, A55205_dir] 
AH = ThermistorDataPlotter(ah_file_paths, delimiter=',')

# Hohlaubgletscher
hs_file_paths = [A551FD_dir, A55203_dir, A55200_dir] 
HS = ThermistorDataPlotter(hs_file_paths, delimiter=',')

In [ ]:
CJ.plot_multiple_temperature_profiles(
    snapshot_time='19.08.2025',
    offsets_list=[A55201_offsets, A55202_offsets],
    depth_files=[depth_CJ1G, depth_CJ2G],
    labels=['CJ1G', 'CJ2G'],
    savepath=output_dir + "CH_profiles.png",
    title="Chessjen Temperature Profiles"
)

### Incorporate the Tynitag thermistor data into the profiles

In [ ]:
# Tynitag thermistor ice temperature dir
tt_icetemp_dir = os.path.join(root_dir, "NTC_tynitag", "temperature_data", "2024") + "/"

# set glacier specific dirs
tt_CJ_dirs  = [CJ1TT_dir, CJ2TT_dir, CJ3TT_dir, CJ4TT_dir]

# read depth files
depths_cj1tt = read_thermistor_depths(depth_CJ1TT)
depths_cj2tt = read_thermistor_depths(depth_CJ2TT)
depths_cj3tt = read_thermistor_depths(depth_CJ3TT)
depths_cj4tt = read_thermistor_depths(depth_CJ4TT)

# generate tynitag thermistor data object
tt_CJ_data = ThermistorData(tt_CJ_dirs, delimiter=",")

In [ ]:
# read chain temperature offsets from CSV file
offsets_path_G = os.path.join(project_root, "data", "calibration", "corrected_chain_offsets.csv")
offsets_path_TT = os.path.join(ICETEMP_ROOT, "NTC_tynitag", "calibration_data", "all_logger_offsets.csv")
corrected_offsets_G = pd.read_csv(offsets_path_G, index_col="chain")
corrected_offsets_TT = pd.read_csv(offsets_path_TT)

In [ ]:
## read out specific borehole data

# Chessjen
CJ1TT_data = CJ1TT.get_ntc_data_with_offsets('7', corrected_offsets_TT, aggregate='all') # average over entire period
CJ2TT_data = CJ2TT.get_ntc_data_with_offsets('8', corrected_offsets_TT, aggregate='all') # average over entire period
CJ3TT_data = CJ3TT.get_ntc_data_with_offsets('14', corrected_offsets_TT, aggregate='all') # average over entire period
CJ4TT_data = CJ4TT.get_ntc_data_with_offsets('15', corrected_offsets_TT, aggregate='all') # average over entire period

# Alphubel
AH1TT_data = AH1TT.get_ntc_data_with_offsets('9', corrected_offsets_TT, aggregate='all') # average over entire period
AH2TT_data = AH2TT.get_ntc_data_with_offsets('10', corrected_offsets_TT, aggregate='all') # average over entire period
AH3TT_data = AH3TT.get_ntc_data_with_offsets('13', corrected_offsets_TT, aggregate='all') # average over entire period

# Hohsaas
HS1TT_data = HS1TT.get_ntc_data_with_offsets('5', corrected_offsets_TT, aggregate='all') # average over entire period
HS2TT_data = HS2TT.get_ntc_data_with_offsets('6', corrected_offsets_TT, aggregate='all') # average over entire period

### Generate dataframes for tynitag data
- with offsets 
- without offsets

In [ ]:
CJ.plot_multiple_temperature_profiles(
    offsets_list=[A55201_offsets, A55202_offsets],
    depth_files=[depth_CJ1G, depth_CJ2G, depth_CJ1TT, depth_CJ2TT, depth_CJ3TT, depth_CJ4TT],
    labels=['CJ1G', 'CJ2G', 'CJ1TT', 'CJ2TT', 'CJ3TT', 'CJ4TT'],
    ntc_data_list=[CJ1TT_data, CJ2TT_data, CJ3TT_data, CJ4TT_data],
    figsize=(3,5),
    dpi=300,
    savepath=output_dir + "CJ_profiles.png",
    xtick_step=1.0,
    use_full_period=True,
    start_time="08.08.2025 12:00:00",
    end_time="15.12.2025 12:00:00",
    zaa_depth=19.8,  # matches panel a's ZAA fit (Chessjen), see notebook 5/6
)

In [ ]:
AH.plot_multiple_temperature_profiles(
    snapshot_time='07.09.2025',
    offsets_list=[A551FE_offsets, A55204_offsets, A55205_offsets],
    depth_files=[depth_AH1G, depth_AH2G, depth_AH3G, depth_AH1TT, depth_AH2TT, depth_AH3TT],
    labels=['AH1G', 'AH2G', 'AH3G', 'AH1TT','AH2TT','AH3TT'],
    ntc_data_list=[AH1TT_data, AH2TT_data, AH3TT_data],
    savepath=output_dir + "AH_profiles.png",
    base_fontsize=12,
    figsize=(3,5),
    dpi=300,
    show_title=False,    # no title on the panel
    use_full_period=True,
    start_time="07.08.2025 12:00:00",
    end_time="07.09.2025 12:00:00",
    zaa_depth=21.7,  # matches panel a's ZAA fit (Alphubel), see notebook 5/6
)


### Hohsaas profiles for drone line 1

In [ ]:
HS.plot_multiple_temperature_profiles(
    snapshot_time='27.09.2025',
    offsets_list=[A551FD_offsets, A55203_offsets, A55200_offsets],
    depth_files=[depth_HS1G, depth_HS2G, depth_HS3G, depth_HS1TT, depth_HS2TT],
    labels=['HS1G', 'HS2G', 'HS3G', 'HS1TT', 'HS2TT'],
    exclude_labels=['HS1G','HS3G'],  # exclude HS1G and HS3G from the plot
    ntc_data_list=[HS1TT_data, HS2TT_data],
    savepath=output_dir + "HS_profiles_drone_1.png",
    dpi=300,
    figsize=(3,5),
    xtick_step=0.5,
)

### Hohsaas profiles for drone line 2

In [ ]:
HS.plot_multiple_temperature_profiles(
    snapshot_time='27.09.2025',
    offsets_list=[A551FD_offsets, A55203_offsets, A55200_offsets],
    depth_files=[depth_HS1G, depth_HS2G, depth_HS3G, depth_HS1TT, depth_HS2TT],
    labels=['HS1G', 'HS2G', 'HS3G', 'HS1TT', 'HS2TT'],
    exclude_labels=['HS1TT','HS2TT'],  # exclude HS1TT and HS2TT from the plot
    ntc_data_list=[HS1TT_data, HS2TT_data],
    savepath=output_dir + "HS_profiles_drone_2.png",
    dpi=300,
    figsize=(3,5)
)

## Generate timeseries temperature heatmaps

In [ ]:
fig, ax, meta_ah1g = plot_chain_temperature_heatmap(
    AH1G,
    start_time="07.08.2025 10:00:00",
    end_time="07.09.2025 12:00:00",
    offsets=A551FE_offsets,
    depth_file=depth_AH1G,
    time_freq="3H",
    depth_step=0.01,
    smooth_time_sigma=1.0,
    smooth_depth_sigma=0.5,
    temp_step=0.2,
    savepath=output_dir + "AH1G_temperature_heatmap.png",
    cbar_min=-3.0,
    bedrock_depth = 50.6
)

plt.show()

In [ ]:
fig, ax, meta_ah2g = plot_chain_temperature_heatmap(
    AH2G,
    start_time="07.08.2025 10:00:00",
    end_time="07.09.2025 12:00:00",
    offsets=A55204_offsets,
    depth_file=depth_AH2G,
    time_freq="3H",
    depth_step=0.01,
    smooth_time_sigma=1.0,
    smooth_depth_sigma=0.5,
    temp_step=0.2,
    savepath=output_dir + "AH2G_temperature_heatmap.png",
    cbar_min=-3.0,
    bedrock_depth = 20.3
)

plt.show()

In [ ]:
fig, ax, meta_ah3g = plot_chain_temperature_heatmap(
    AH3G,
    start_time="07.08.2025 10:00:00",
    end_time="07.09.2025 12:00:00",
    offsets=A55205_offsets,
    depth_file=depth_AH3G,
    time_freq="3H",
    depth_step=0.01,
    smooth_time_sigma=1.0,
    smooth_depth_sigma=0.5,
    temp_step=0.2,
    savepath=output_dir + "AH3G_temperature_heatmap.png",
    cbar_min=-3.0,
    bedrock_depth=58.3
)

plt.show()

### Build mosaic figure for Alphubel South

In [ ]:
# Build the mosaic with one shared colorbar and panel tags
fig_m, (axs_heat, axs_ts) = mosaic_chain_heatmaps(
    [meta_ah1g, meta_ah2g, meta_ah3g],
    titles=["AH1G", "AH2G", "AH3G"],
    panel_tags=("a","b","c"),
    figsize=(19, 10),      # taller to fit the second row
    two_rows=True,        # enable 2x3 layout with time series
    line_width=1.9,
    line_alpha=0.9,
    line_color_base=plt.cm.viridis,
    contour_kwargs={"colors": "black", "linewidths": 1.0, "alpha": 0.35},
    savepath=output_dir + "mosaic_AH1G_AH2G_AH3G.pdf",
    ts_y_limits=((-1.0,0.1),(-3.5,0.1),(-1.0,0.1)),
    ts_y_tick_steps=[0.2,1.0,0.2]
)

plt.show()

# save figure
fig_m.savefig(output_dir + "mosaic_AH1G_AH2G_AH3G.pdf", dpi=300, bbox_inches='tight')

fig_m.savefig(os.path.join(project_root, "figures", "supplement", "figS14_mosaic_AH1G_AH2G_AH3G.pdf"), dpi=300, bbox_inches='tight')


In [ ]:
fig, ax, meta_hs1g = plot_chain_temperature_heatmap(
    HS1G,
    start_time="15.08.2025 12:00:00",
    end_time="27.09.2025 12:00:00",
    offsets=A551FD_offsets,          # or None if you do not want corrections
    depth_file=depth_HS1G,           # or None if you do not have a depth file
    time_freq="3H",                  # adjust if you want finer / coarser temporal grid
    depth_step=0.01,                 # vertical interpolation step (m)
    smooth_time_sigma=1.0,           # light temporal smoothing (in 3H steps here)
    smooth_depth_sigma=0.5,          # light vertical smoothing
    temp_step=0.2,                   # color level step (°C)
    savepath=output_dir + "HS1G_temperature_heatmap.png",
    cbar_min=-1.5,
    bedrock_depth=29,
)

plt.show()

In [ ]:
fig, ax, meta_hs2g = plot_chain_temperature_heatmap(
    HS2G,
    start_time="15.08.2025 12:00:00",
    end_time="27.09.2025 12:00:00",
    offsets=A55203_offsets,
    depth_file=depth_HS2G,
    time_freq="3H",
    depth_step=0.01,
    smooth_time_sigma=1.0,
    smooth_depth_sigma=0.5,
    temp_step=0.2,
    savepath=output_dir + "HS2G_temperature_heatmap.png",
    cbar_min=-1.5,
    bedrock_depth=21.5
)

plt.show()


In [ ]:
fig, ax, meta_hs3g = plot_chain_temperature_heatmap(
    HS3G,
    start_time="15.08.2025 12:00:00",
    end_time="27.09.2025 12:00:00",
    offsets=A55200_offsets,
    depth_file=depth_HS3G,
    time_freq="3H",
    depth_step=0.01,
    smooth_time_sigma=1.0,
    smooth_depth_sigma=0.5,
    temp_step=0.2,
    savepath=output_dir + "HS3G_temperature_heatmap.png",
    cbar_min=-1.5,
    bedrock_depth=31
)

plt.show()


In [ ]:
# Build the mosaic with one shared colorbar and panel tags
fig_m, (axs_heat, axs_ts) = mosaic_chain_heatmaps(
    [meta_hs1g, meta_hs2g, meta_hs3g],
    titles=["HS1G", "HS2G", "HS3G"],
    panel_tags=("a","b","c"),
    figsize=(19, 10),      # taller to fit the second row
    two_rows=True,        # enable 2x3 layout with time series
    line_width=1.9,
    line_alpha=0.9,
    line_color_base=plt.cm.viridis,
    contour_kwargs={"colors": "black", "linewidths": 1.0, "alpha": 0.35},
    savepath=output_dir + "mosaic_HS1G_HS2G_HS3G.pdf",
    ts_y_limits=((-1.5,0.0),(-1.5,0.0),(-1.5,0.0)),
    ts_y_tick_steps=[0.5,0.5,0.5]
)

plt.show()

# save figure
fig_m.savefig(output_dir + "mosaic_HS1G_HS2G_HS3G.pdf", dpi=300, bbox_inches='tight')

fig_m.savefig(os.path.join(project_root, "figures", "supplement", "figS16_mosaic_HS1G_HS2G_HS3G.pdf"), dpi=300, bbox_inches='tight')

In [ ]:
fig, ax, meta_cj1g = plot_chain_temperature_heatmap(
    CJ1G,
    start_time="08.08.2025 12:00:00",
    end_time="15.12.2025 12:00:00",
    offsets=A55201_offsets,
    depth_file=depth_CJ1G,
    time_freq="3H",
    depth_step=0.01,
    smooth_time_sigma=1.0,
    smooth_depth_sigma=0.5,
    temp_step=0.2,
    savepath=output_dir + "CJ1G_temperature_heatmap.png",
    cbar_min=-2.0,
    bedrock_depth=38.3
)

plt.show()


In [ ]:
fig, ax, meta_cj2g = plot_chain_temperature_heatmap(
    CJ2G,
    start_time="08.08.2025 12:00:00",
    end_time="15.12.2025 12:00:00",
    offsets=A55202_offsets,
    depth_file=depth_CJ2G,
    time_freq="3H",
    depth_step=0.01,
    smooth_time_sigma=1.0,
    smooth_depth_sigma=0.5,
    temp_step=0.2,
    savepath=output_dir + "CJ2G_temperature_heatmap.png",
    cbar_min=-2.0,
    bedrock_depth=17.0
)

plt.show()


In [ ]:
fig_m, (axs_heat, axs_ts) = mosaic_chain_heatmaps(
    [meta_cj1g, meta_cj2g],
    titles=["CJ1G", "CJ2G"],
    panel_tags=("a","b"),
    figsize=(12, 10),
    two_rows=True,
    line_width=1.8,
    line_alpha=0.9,
    line_color_base=plt.cm.viridis,
    contour_kwargs={"colors": "black", "linewidths": 1.0, "alpha": 0.35},
    savepath=output_dir + "mosaic_CJ1G_CJ2G.pdf",
    ts_y_limits=((-1.5,0.5),(-1.5,0.5)),
    ts_y_tick_steps=[0.5,0.5]
)

plt.show()

# save figure
fig_m.savefig(output_dir + "mosaic_CJ1G_CJ2G.pdf", dpi=300, bbox_inches='tight')

fig_m.savefig(os.path.join(project_root, "figures", "supplement", "figS15_mosaic_CJ1G_CJ2G.pdf"), dpi=300, bbox_inches='tight')

In [ ]:
# Flatten all offsets into a single array, ignoring NaNs
all_offsets = corrected_offsets_G.values.flatten()
all_offsets = all_offsets[~pd.isnull(all_offsets)]

plt.figure(figsize=(7, 4))
plt.boxplot(all_offsets, vert=False, patch_artist=True,
            boxprops=dict(facecolor='lightgray', color='black'),
            medianprops=dict(color='red', linewidth=2))
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Offset [°C]')
plt.title('Spread of Geoprecision Chain Offsets Around 0°C')

### Compute geprecision temperature statistics

In [ ]:
# After creating the heatmap (which generates meta_ah1g):
fig, ax, meta_ah1g = plot_chain_temperature_heatmap(
    AH1G,
    start_time="07.08.2025 10:00:00",
    end_time="07.09.2025 12:00:00",
    offsets=A551FE_offsets,
    depth_file=depth_AH1G,
    # ...other params...
)

# Now use the meta dict for statistics:
fig_stats, stats_df = plot_gp_statistics(
    meta_ah1g,
    savepath=output_dir + "AH1G_statistics.png",
    title="AH1G Temperature Statistics",
    seasonal_months=[(6,7,8), (12,1,2)],  # summer/winter
    equilibration_days=4,
    show_boxplot=True,
    show_table=True
)

plt.show()

In [ ]:
fig_stats_ah2g, stats_df_ah2g = plot_gp_statistics(
    meta_ah2g,
    savepath=output_dir + "AH2G_statistics.png",
    title="AH2G Temperature Statistics",
    seasonal_months=[(6,7,8), (12,1,2)],
    equilibration_days=4,
    show_boxplot=True,
    show_table=True
)

plt.show()


In [ ]:
fig_stats_ah3g, stats_df_ah3g = plot_gp_statistics(
    meta_ah3g,
    savepath=output_dir + "AH3G_statistics.png",
    title="AH3G Temperature Statistics",
    seasonal_months=[(6,7,8), (12,1,2)],
    equilibration_days=4,
    show_boxplot=True,
    show_table=True
)

plt.show()

In [ ]:
fig_stats_cj1g, stats_df_cj1g = plot_gp_statistics(
    meta_cj1g,
    savepath=output_dir + "CJ1G_statistics.png",
    title="CJ1G Temperature Statistics",
    seasonal_months=[(6,7,8), (12,1,2)],
    equilibration_days=4,
    show_boxplot=True,
    show_table=True
)

plt.show()

In [ ]:
fig_stats_hs1g, stats_df_hs1g = plot_gp_statistics(
    meta_hs1g,
    savepath=output_dir + "HS1G_statistics.png",
    title="HS1G Temperature Statistics",
    seasonal_months=[(6,7,8), (12,1,2)],
    equilibration_days=4,
    show_boxplot=True,
    show_table=True
)

plt.show()

In [ ]:
fig_stats_hs2g, stats_df_hs2g = plot_gp_statistics(
    meta_hs2g,
    savepath=output_dir + "HS2G_statistics.png",
    title="HS2G Temperature Statistics",
    seasonal_months=[(6,7,8), (12,1,2)],
    equilibration_days=4,
    show_boxplot=True,
    show_table=True
)

plt.show()

In [ ]:
fig_stats_hs3g, stats_df_hs3g = plot_gp_statistics(
    meta_hs3g,
    savepath=output_dir + "HS3G_statistics.png",
    title="HS3G Temperature Statistics",
    seasonal_months=[(6,7,8), (12,1,2)],
    equilibration_days=4,
    show_boxplot=True,
    show_table=True
)

plt.show()

## Overview: Measurement Periods

In [ ]:
# ── helpers ───────────────────────────────────────────────────────────────────
def _chain_extent(td_obj):
    """Return (first_time, last_time) for a Geoprecision chain."""
    df = td_obj.get_chain_data()
    valid = df['TIME'].dropna()
    return (valid.min(), valid.max()) if not valid.empty else (None, None)

def _ntc_extent(td_obj):
    """Return (first_time, last_time) for a TyniTag NTC logger."""
    df = td_obj.get_ntc_data()
    valid = df['TIME'].dropna()
    return (valid.min(), valid.max()) if not valid.empty else (None, None)

# ── TyniTag-only glaciers (Sex Rouge, Tortin, Corvatsch) ─────────────────────
_tt_base = TT_icetemp_dir   # already defined in notebook

SR1TT = ThermistorData(_tt_base + "SR1TT_20240806_20250724_spliced.csv", ",")
SR2TT = ThermistorData(_tt_base + "SR2TT_20240806_20250724_spliced.csv", ",")
GT1TT = ThermistorData(_tt_base + "GT1TT_20240807_20250723_spliced.csv", ",")
GT2TT = ThermistorData(_tt_base + "GT2TT_20240807_20250723_spliced.csv", ",")
CV1TT = ThermistorData(_tt_base + "CT1TT_20240828_20250905.csv", ",")
CV2TT = ThermistorData(_tt_base + "CT2TT_20240828_20250905_spliced.csv", ",")

# ── collect periods ───────────────────────────────────────────────────────────
sensors = []

# Alphubel – GP chains
for label, obj in [("AH1G", AH1G), ("AH2G", AH2G), ("AH3G", AH3G)]:
    s, e = _chain_extent(obj)
    sensors.append(dict(label=label, glacier="Alphubel", sensor_type="Geoprecision", start=s, end=e))
# Alphubel – TyniTag
for label, obj in [("AH1TT", AH1TT), ("AH2TT", AH2TT), ("AH3TT", AH3TT)]:
    s, e = _ntc_extent(obj)
    sensors.append(dict(label=label, glacier="Alphubel", sensor_type="TyniTag", start=s, end=e))

# Hohsaas – GP chains
for label, obj in [("HS1G", HS1G), ("HS2G", HS2G), ("HS3G", HS3G)]:
    s, e = _chain_extent(obj)
    sensors.append(dict(label=label, glacier="Hohsaas", sensor_type="Geoprecision", start=s, end=e))
# Hohsaas – TyniTag
for label, obj in [("HS1TT", HS1TT), ("HS2TT", HS2TT)]:
    s, e = _ntc_extent(obj)
    sensors.append(dict(label=label, glacier="Hohsaas", sensor_type="TyniTag", start=s, end=e))

# Chessjen – GP chains
for label, obj in [("CJ1G", CJ1G), ("CJ2G", CJ2G)]:
    s, e = _chain_extent(obj)
    sensors.append(dict(label=label, glacier="Chessjen", sensor_type="Geoprecision", start=s, end=e))
# Chessjen – TyniTag
for label, obj in [("CJ1TT", CJ1TT), ("CJ2TT", CJ2TT), ("CJ3TT", CJ3TT), ("CJ4TT", CJ4TT)]:
    s, e = _ntc_extent(obj)
    sensors.append(dict(label=label, glacier="Chessjen", sensor_type="TyniTag", start=s, end=e))

# Sex Rouge – TyniTag only
for label, obj in [("SR1TT", SR1TT), ("SR2TT", SR2TT)]:
    s, e = _ntc_extent(obj)
    sensors.append(dict(label=label, glacier="Sex Rouge", sensor_type="TyniTag", start=s, end=e))

# Tortin – TyniTag only
for label, obj in [("GT1TT", GT1TT), ("GT2TT", GT2TT)]:
    s, e = _ntc_extent(obj)
    sensors.append(dict(label=label, glacier="Tortin", sensor_type="TyniTag", start=s, end=e))

# Corvatsch – TyniTag only
for label, obj in [("CV1TT", CV1TT), ("CV2TT", CV2TT)]:
    s, e = _ntc_extent(obj)
    sensors.append(dict(label=label, glacier="Corvatsch", sensor_type="TyniTag", start=s, end=e))

df_periods = pd.DataFrame(sensors)
print(df_periods[["label","glacier","sensor_type","start","end"]].to_string(index=False))


In [ ]:
# ── global font: Arial ────────────────────────────────────────────────────────

# ── sensor type colours ───────────────────────────────────────────────────────
COLOR_GP  = "#C87070"   # muted brick-red – Geoprecision chain
COLOR_TT  = "#FFFFFF"   # slightly darker than pure white – TyniTag NTC
EDGE_TT   = "#555555"   # dark edge for the light bars

SENSOR_ALPHA = {"Geoprecision": 0.85, "TyniTag": 1.0}
BAR_HEIGHT    = 0.55

glacier_order = ["Alphubel", "Hohsaas", "Chessjen", "Sex Rouge", "Tortin", "Corvatsch"]

# ── build ordered y-axis (GP before TT within each group) ────────────────────
df_sorted = pd.concat(
    [df_periods[df_periods["glacier"] == g].sort_values(
        ["sensor_type", "label"], ascending=[True, True])
     for g in glacier_order if g in df_periods["glacier"].values],
    ignore_index=True
)
df_sorted = df_sorted[::-1].reset_index(drop=True)   # invert so first glacier is on top

fig, ax = plt.subplots(figsize=(11, 7))

group_bounds = {}

for i, row in df_sorted.iterrows():
    if pd.isna(row["start"]) or pd.isna(row["end"]):
        continue
    is_gp = row["sensor_type"] == "Geoprecision"
    color     = COLOR_GP if is_gp else COLOR_TT
    edgecolor = "black"
    alpha     = SENSOR_ALPHA[row["sensor_type"]]
    lw        = 0.8
    hatch     = None if is_gp else ""

    ax.barh(
        i, (row["end"] - row["start"]).days, left=row["start"],
        height=BAR_HEIGHT, color=color, alpha=alpha,
        linewidth=lw, edgecolor=edgecolor, hatch=hatch,
        zorder=3
    )

    if row["glacier"] not in group_bounds:
        group_bounds[row["glacier"]] = [i, i]
    else:
        group_bounds[row["glacier"]][0] = min(group_bounds[row["glacier"]][0], i)
        group_bounds[row["glacier"]][1] = max(group_bounds[row["glacier"]][1], i)

# ── uniform light background bands per glacier group ─────────────────────────
for glacier in glacier_order:
    if glacier not in group_bounds:
        continue
    y0, y1 = group_bounds[glacier]

# ── fix ylim to exactly match band extent ─────────────────────────────────────
ax.set_ylim(-0.5, len(df_sorted) - 0.5)

# ── borehole y-tick labels ────────────────────────────────────────────────────
ax.set_yticks(range(len(df_sorted)))
ax.set_yticklabels(df_sorted["label"], fontsize=12)
ax.tick_params(axis="y", length=0, pad=4)

# ── glacier group labels with square-bracket connectors ───────────────────────
trans    = ax.get_yaxis_transform()
x_label  = -0.100  # glacier name (right-aligned)
x_bar    = -0.085  # vertical bar
x_serif  = -0.075  # serif tips
serif_h  = 0.38    # vertical reach of serifs (data units)

for glacier in glacier_order:
    if glacier not in group_bounds:
        continue
    y0, y1 = group_bounds[glacier]
    y_mid  = (y0 + y1) / 2

    ax.text(x_label, y_mid, glacier,
            transform=trans, ha="right", va="center",
            fontsize=13, fontweight="bold", color="0.2")

    ax.plot([x_bar, x_bar], [y0 - serif_h, y1 + serif_h],
            transform=trans, color="black", linewidth=0.9, clip_on=False)
    ax.plot([x_bar, x_serif], [y1 + serif_h, y1 + serif_h],
            transform=trans, color="black", linewidth=0.9, clip_on=False)
    ax.plot([x_bar, x_serif], [y0 - serif_h, y0 - serif_h],
            transform=trans, color="black", linewidth=0.9, clip_on=False)

# ── x-axis (time) ─────────────────────────────────────────────────────────────
x_min = df_periods["start"].dropna().min() - pd.Timedelta(days=14)
x_max = max(df_periods["end"].dropna().max() + pd.Timedelta(days=7),
            pd.Timestamp("2025-11-07"))
ax.set_xlim(x_min, x_max)

ax.xaxis.set_major_locator(mdates.MonthLocator())

def _month_fmt(x, pos):
    dt = mdates.num2date(x)
    return dt.strftime("%b\n%Y") if dt.month == 1 else dt.strftime("%b")

ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(_month_fmt))
ax.tick_params(axis="x", which="major", labelsize=11.5)

ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.7)

# ── mass balance measurement periods ─────────────────────────────────────────
MB_STAKE_COLOR = "#E8916A"   # pale reddish-orange – ablation stake readings
MB_SNOW_COLOR  = "#A8CADF"   # pale blue  – snow depth measurements
MB_STAKE_ALPHA = 0.4
MB_SNOW_ALPHA  = 0.4

mb_periods = [
    {"start": pd.Timestamp("2024-08-01"), "end": pd.Timestamp("2024-10-31"),
     "color": MB_STAKE_COLOR},
    {"start": pd.Timestamp("2025-08-01"), "end": pd.Timestamp("2025-10-31"),
     "color": MB_STAKE_COLOR},
    {"start": pd.Timestamp("2025-04-01"), "end": pd.Timestamp("2025-05-31"),
     "color": MB_SNOW_COLOR},
]

for period in mb_periods:
    alpha = MB_STAKE_ALPHA if period["color"] == MB_STAKE_COLOR else MB_SNOW_ALPHA
    ax.axvspan(period["start"], period["end"],
               color=period["color"], alpha=alpha, zorder=2)

# ── GPR campaign markers ──────────────────────────────────────────────────────
GPR_COLOR     = "black"
GPR_UAV_COLOR = "black"
GPR_LW        = 2.8

gpr_campaigns = [
    {"date": pd.Timestamp("2025-05-16"), "glaciers": ["Alphubel"],
     "color": GPR_COLOR, "label": "GPR survey"},
    {"date": pd.Timestamp("2024-08-06"),  "glaciers": ["Hohsaas", "Chessjen", "Sex Rouge", "Tortin"],
     "color": GPR_COLOR, "label": "GPR survey"},
    {"date": pd.Timestamp("2024-08-30"), "glaciers": ["Corvatsch"],
     "color": GPR_COLOR, "label": "GPR survey"},
    {"date": pd.Timestamp("2025-09-28"), "glaciers": ["Hohsaas"],
     "color": GPR_UAV_COLOR, "label": "UAV GPR survey"},
]

for camp in gpr_campaigns:
    color = camp["color"]
    y_vals = []
    for g in camp["glaciers"]:
        if g in group_bounds:
            y_vals.extend(group_bounds[g])
    if not y_vals:
        continue
    y_lo = min(y_vals) - 0.5
    y_hi = max(y_vals) + 0.5

    ax.plot([camp["date"], camp["date"]], [y_lo, y_hi],
            color=color, lw=GPR_LW, ls="--", zorder=4)

    ax.text(camp["date"] + pd.Timedelta(days=4), y_hi - 0.4,
            camp["label"], rotation=0,
            va="top", ha="left", fontsize=11, color=color, zorder=5,
            fontweight="bold")

# ── legend ────────────────────────────────────────────────────────────────────
patch_gp    = mpatches.Patch(facecolor=COLOR_GP, alpha=0.85, edgecolor="black",
                              label="Geoprecision")
patch_tt    = mpatches.Patch(facecolor=COLOR_TT, edgecolor=EDGE_TT, label="Tinytag")
patch_stake = mpatches.Patch(facecolor=MB_STAKE_COLOR, alpha=MB_STAKE_ALPHA,
                              edgecolor=MB_STAKE_COLOR, label="Ablation stake readings")
patch_snow  = mpatches.Patch(facecolor=MB_SNOW_COLOR, alpha=MB_SNOW_ALPHA,
                              edgecolor=MB_SNOW_COLOR, label="Snow depth measurements")
ax.legend(handles=[patch_gp, patch_tt, patch_stake, patch_snow],
          loc="upper left", fontsize=11.5,
          framealpha=0.9, edgecolor="black", fancybox=False)

ax.set_xlabel("Date", fontsize=13.5)

fig.subplots_adjust(left=0.18, right=0.97, top=0.97, bottom=0.10)

plt.savefig(os.path.join(project_root, "figures", "supplement", "figS02_deployment_timeline.pdf"), dpi=300, bbox_inches="tight")
plt.savefig(output_dir + "deployment_timeline.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved to", os.path.join(project_root, "figures", "supplement", "figS02_deployment_timeline.pdf"))

---

## Part 2 — Historical Comparison

## Sex Rouge: Comparison with Historical Data (Fischer 2018)

Vertical englacial temperature profiles from Fischer (2018) for two boreholes on Glacier du Sex Rouge (Tables A6, A7 in dissertation), compared against 2024/25 mean temperatures from SR1TT and SR2TT.

**BH1** (2805 m a.s.l.): 40 m thermistor chain with 10 active sensors — four measurement dates between Nov 2013 and Sep 2015.  
**BH2** (2777 m a.s.l.): Single thermistor at ~9–10 m depth — two measurement dates (Sep 2014, Jul 2015).

Note: the GLENGLAT database entry previously attributed to Signer (2014) corresponds to the Nov 2013 and Sep 2014 BH1 snapshots from Fischer (2018).

In [ ]:
# ── Fischer (2018) Table A6 — Borehole 1 (2805 m a.s.l., CH1903: 582609/130602)
# 10 thermistors (T6–T15); ice surface ablates ~0.4 m yr⁻¹ between surveys
fischer_bh1 = {
    '17 Nov 2013': {
        'depth': [1.9, 3.9, 4.9, 5.9, 7.9, 9.9, 14.9, 19.9, 24.9, 34.9],
        'temp':  [-0.51, -0.72, -0.82, -0.90, -0.95, -0.88, -0.52, -0.12, -0.05, -0.05],
    },
    '11 Sep 2014': {
        'depth': [1.5, 3.5, 4.5, 5.5, 7.5, 9.5, 14.5, 19.5, 24.5, 34.5],
        'temp':  [-0.38, -0.88, -1.00, -1.08, -1.07, -0.95, -0.55, -0.17, -0.06, -0.06],
    },
    '12 Jul 2015': {
        'depth': [1.0, 3.0, 4.0, 5.0, 7.0, 9.0, 14.0, 19.0, 24.0, 34.0],
        'temp':  [-0.53, -1.25, -1.45, -1.53, -1.38, -1.26, -0.54, -0.15,  0.02, -0.03],
    },
    '21 Sep 2015': {
        # T6 melted out at surface — 9 readings
        'depth': [0.6, 1.6, 2.6, 4.6, 6.6, 11.6, 16.6, 21.6, 31.6],
        'temp':  [-0.90, -0.39, -0.67, -0.96, -1.12, -0.60, -0.18,  0.02, -0.03],
    },
}

# ── Fischer (2018) Table A7 — Borehole 2 (2777 m a.s.l., CH1903: 582580/130750)
# Single thermistor at bottom only
fischer_bh2 = [
    {'date': '11 Sep 2014', 'depth': 9.70, 'temp': -1.27},
    {'date': '12 Jul 2015', 'depth': 9.05, 'temp': -1.38},
]

# ── Our 2024/25 measurements (stats_sr.csv, full-period statistics) ──
# SR1TT (~2810 m, near BH1): white = 10.0 m, black = 15.2 m
sr1tt_depths = [10.0, 15.2]
sr1tt_means  = [-0.397, -0.273]
sr1tt_mins   = [-0.666, -0.403]
sr1tt_maxs   = [ 0.000, -0.194]  # white probe: max capped at 0 °C

# SR2TT (~2760 m, near BH2): white = 10.0 m, black = 14.4 m
sr2tt_depths = [10.0, 14.4]
sr2tt_means  = [-0.348, -0.300]
sr2tt_mins   = [-0.672, -0.411]
sr2tt_maxs   = [-0.038, -0.176]

# ── Colours ───────────────────────────────────────────────────────────────────
SR_COLOR = '#C0392B'
BH1_COLORS = ['#BBBBBB', '#888888', '#555555', '#222222']
BH2_COLORS = ['#888888', '#444444']
BH2_MARKERS = ['s', 'D']

LABEL_KWARGS = dict(
    fontsize=13, fontweight='bold', va='top', ha='left',
    bbox=dict(facecolor='white', edgecolor='none', pad=2.0)
)
LEGEND_KWARGS = dict(fontsize=8.5, loc='lower left', fancybox=False, edgecolor='black')

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 6), dpi=150, sharey=True)
fig.subplots_adjust(wspace=0.06)

# ── Left panel: BH1 (four profiles) + SR1TT ──────────────────────────────────
ax = axes[0]
ax.axvline(0, color='black', lw=1.5, ls='--', zorder=1)

for (date, prof), color in zip(fischer_bh1.items(), BH1_COLORS):
    ax.plot(prof['temp'], prof['depth'], color=color, lw=1.8,
            marker='o', ms=4, label=f'Fischer (2018) BH1 — {date}', zorder=3)

ax.errorbar(sr1tt_means, sr1tt_depths,
            xerr=[[m - lo for m, lo in zip(sr1tt_means, sr1tt_mins)],
                  [min(hi, 0.0) - m for m, hi in zip(sr1tt_means, sr1tt_maxs)]],
            fmt='D', color=SR_COLOR, ecolor=SR_COLOR,
            elinewidth=1.5, capsize=4, ms=7,
            label='SR1TT (2024/25, mean [min–max])', zorder=4)

ax.set_title('SR1TT (~2805 m a.s.l.)', fontsize=12, pad=8)
ax.set_xlabel('Temperature (°C)', fontsize=11)
ax.set_ylabel('Depth (m)', fontsize=11)
ax.set_xlim(-1.75, 0.15)
ax.invert_yaxis()
ax.set_ylim(37, 0)
ax.grid(True, alpha=0.3, lw=0.5)
ax.legend(**LEGEND_KWARGS)
ax.tick_params(labelsize=10)
ax.text(0.02, 0.98, '(a)', transform=ax.transAxes, **LABEL_KWARGS)

# ── Right panel: BH2 (2 single-point measurements) + SR2TT ──────────────────
ax = axes[1]
ax.axvline(0, color='black', lw=1.5, ls='--', zorder=1)

for meas, color, marker in zip(fischer_bh2, BH2_COLORS, BH2_MARKERS):
    ax.plot(meas['temp'], meas['depth'], marker=marker, color=color,
            ms=8, ls='none',
            label=f"Fischer (2018) BH2 — {meas['date']}", zorder=3)

ax.errorbar(sr2tt_means, sr2tt_depths,
            xerr=[[m - lo for m, lo in zip(sr2tt_means, sr2tt_mins)],
                  [min(hi, 0.0) - m for m, hi in zip(sr2tt_means, sr2tt_maxs)]],
            fmt='D', color=SR_COLOR, ecolor=SR_COLOR,
            elinewidth=1.5, capsize=4, ms=7,
            label='SR2TT (2024/25, mean [min–max])', zorder=4)

ax.set_title('SR2TT (~2777 m a.s.l.)', fontsize=12, pad=8)
ax.set_xlabel('Temperature (°C)', fontsize=11)
ax.set_xlim(-1.75, 0.15)
ax.invert_yaxis()
ax.set_ylim(37, 0)
ax.grid(True, alpha=0.3, lw=0.5)
ax.legend(**LEGEND_KWARGS)
ax.tick_params(labelsize=10)
ax.text(0.02, 0.98, '(b)', transform=ax.transAxes, **LABEL_KWARGS)

out_path = output_dir + 'sex_rouge_historical_comparison.pdf'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')


## Corvatsch: Comparison with Historical Data (Haeberli et al. 2004)

Englacial temperature profiles from the GLENGLAT database (borehole id 92, source: haeberli2004) for Vadret dal Corvatsch (~3340 m a.s.l.), compared against 2024/25 mean temperatures from CV2TT.

Five measurement dates from Sep 1999 to Sep 2000; data were digitized from figures in Haeberli et al. (2004, *Journal of Glaciology*, 50(168), 129–136). CV2TT corresponds to former Borehole 12 on the glacier.


In [ ]:
# ── GLENGLAT borehole 92: Vadret dal Corvatsch (Haeberli et al. 2004)
# 3340 m a.s.l.; 5 profiles Sep 1999 – Sep 2000, data digitized from publication
haeberli_profiles = {
    '14 Sep 1999': {
        'depth': [2.485, 3.040, 3.310, 3.562, 3.831, 4.387, 4.909, 5.464, 5.986,
                  6.576, 7.115, 7.654, 8.479, 9.052, 9.860, 10.433, 10.938, 11.528,
                  12.017, 12.640, 12.910],
        'temp':  [-0.059, -0.876, -1.081, -1.353, -1.660, -2.137, -2.580, -2.956,
                  -3.195, -3.571, -3.844, -4.050, -4.290, -4.360, -4.499, -4.501,
                  -4.537, -4.471, -4.438, -4.373, -4.374],
    },
    '1 Dec 1999': {
        'depth': [1.026, 1.397, 1.834, 2.171, 2.760, 3.382, 3.887, 4.358, 5.014,
                  5.923, 6.495, 7.084, 7.976, 9.423, 11.139, 12.216, 12.957],
        'temp':  [-5.322, -5.186, -5.153, -4.847, -4.203, -3.559, -3.186, -3.051,
                  -2.881, -2.881, -2.847, -2.949, -3.220, -3.492, -3.831, -3.864, -3.831],
    },
    '5 Apr 2000': {
        'depth': [0.976, 2.204, 3.248, 5.486, 7.252, 8.716, 10.382, 12.065, 12.974],
        'temp':  [-8.068, -7.898, -7.356, -6.169, -5.085, -4.542, -4.203, -3.864, -3.763],
    },
    '19 Jun 2000': {
        'depth': [0.993, 2.036, 3.079, 4.005, 4.998, 6.832, 7.976, 8.986, 10.046,
                  11.661, 13.024],
        'temp':  [-0.373, -0.508, -2.712, -4.169, -4.983, -5.390, -5.288, -5.017,
                  -4.983, -4.576, -4.305],
    },
    '20 Sep 2000': {
        'depth': [1.010, 1.565, 2.188, 2.726, 3.786, 4.930, 5.822, 6.916, 8.161,
                  8.868, 10.433, 12.115, 12.974],
        'temp':  [-1.390, -1.119, -0.983, -1.051, -1.763, -2.983, -3.356, -3.492,
                  -3.695, -3.966, -4.339, -4.576, -4.271],
    },
}

# ── Load CV2TT data (logger #12, depth file from thermistor settings) ─────────
# CV2TT = former Borehole 12; initial depths: white=4.3 m, black=9.3 m
# NOTE: black probe failed ~17 h after installation; white probe covers full period
depth_CV1TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_cv2tt.csv")
CV1TT_hist = ThermistorData(TT_icetemp_dir + "CT2TT_20240828_20250905_spliced.csv", ",", depth_CV1TT)

cv1tt_ts = CV1TT_hist.get_ntc_data_with_offsets('12', corrected_offsets_TT, aggregate=None)

# Compute full-period statistics for each probe
cv1tt_white_mean = cv1tt_ts['White Probe Temperature'].mean()
cv1tt_white_min  = cv1tt_ts['White Probe Temperature'].min()
cv1tt_white_max  = cv1tt_ts['White Probe Temperature'].max()
cv1tt_black_mean = cv1tt_ts['Black Probe Temperature'].mean()
cv1tt_black_min  = cv1tt_ts['Black Probe Temperature'].min()
cv1tt_black_max  = cv1tt_ts['Black Probe Temperature'].max()
cv1tt_white_std  = cv1tt_ts['White Probe Temperature'].std()
cv1tt_black_std  = cv1tt_ts['Black Probe Temperature'].std()

# Representative depths: initial deployment (07 Aug 2024): white=4.3 m, black=9.3 m
# Black probe failed after ~17 h — stats represent installation readings only
cv1tt_depths = [4.3, 9.3]
cv1tt_means  = [cv1tt_white_mean, cv1tt_black_mean]
cv1tt_mins   = [cv1tt_white_min,  cv1tt_black_min]
cv1tt_maxs   = [cv1tt_white_max,  cv1tt_black_max]
cv1tt_stds   = [cv1tt_white_std,  cv1tt_black_std]

# ── Colours ───────────────────────────────────────────────────────────────────
CV_COLOR  = '#C0392B'
HAE_COLORS = ['#DDDDDD', '#AAAAAA', '#777777', '#444444', '#111111']

LABEL_KWARGS = dict(
    fontsize=13, fontweight='bold', va='top', ha='left',
    bbox=dict(facecolor='white', edgecolor='none', pad=2.0)
)
LEGEND_KWARGS = dict(fontsize=8.5, loc='lower left', fancybox=False, edgecolor='black')

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(6, 7), dpi=150)

ax.axvline(0, color='black', lw=1.5, ls='--', zorder=1)

for (date, prof), color in zip(haeberli_profiles.items(), HAE_COLORS):
    ax.plot(prof['temp'], prof['depth'], color=color, lw=1.8,
            marker='o', ms=4, label=f'Haeberli et al. (2004) — {date}', zorder=3)

ax.errorbar(cv1tt_means, cv1tt_depths,
            xerr=[[m - lo for m, lo in zip(cv1tt_means, cv1tt_mins)],
                  [hi - m  for m, hi in zip(cv1tt_means, cv1tt_maxs)]],
            fmt='D', color=CV_COLOR, ecolor=CV_COLOR,
            elinewidth=1.5, capsize=4, ms=7,
            label='CV2TT (2024/25, mean [min–max])', zorder=4)

ax.set_title('CV2TT (~3340 m a.s.l.)', fontsize=12, pad=8)
ax.set_xlabel('Temperature (°C)', fontsize=11)
ax.set_ylabel('Depth (m)', fontsize=11)
ax.set_xlim(-9.5, 0.5)
ax.invert_yaxis()
ax.set_ylim(14, 0)
ax.grid(True, alpha=0.3, lw=0.5)
ax.legend(**LEGEND_KWARGS)
ax.tick_params(labelsize=10)
ax.text(0.02, 0.98, '(a)', transform=ax.transAxes, **LABEL_KWARGS)

out_path = output_dir + 'corvatsch_historical_comparison.pdf'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')
print(f'CV2TT white ({cv1tt_depths[0]} m): {cv1tt_white_mean:.3f} [min {cv1tt_white_min:.3f}, max {cv1tt_white_max:.3f}] °C')
print(f'CV2TT black ({cv1tt_depths[1]} m): {cv1tt_black_mean:.3f} [min {cv1tt_black_min:.3f}, max {cv1tt_black_max:.3f}] °C  (NOTE: ~17 h only)')


In [ ]:
# ── Shared style constants ────────────────────────────────────────────────────
SR_COLOR  = '#C0392B'
CV_COLOR  = '#C0392B'
BH1_COLORS  = ['#BBBBBB', '#888888', '#555555', '#222222']
BH2_COLORS  = ['#888888', '#444444']
BH2_MARKERS = ['s', 'D']
HAE_COLORS  = ['#DDDDDD', '#AAAAAA', '#777777', '#444444', '#111111']

LABEL_KWARGS = dict(
    fontsize=17, fontweight='bold', va='top', ha='left',
    bbox=dict(facecolor='white', edgecolor='none', pad=2.0)
)
GLNAME_KWARGS = dict(
    fontsize=14, fontweight='bold', va='bottom', ha='left',
    bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.4')
)
LEG_KW = dict(fontsize=12, fancybox=False, edgecolor='black',
              loc='upper left', bbox_to_anchor=(0, -0.14),
              borderaxespad=0)

def plot_range(ax, means, depths, mins, maxs, color, mean_marker='D', label=None):
    """Plot mean marker with a min–max range line and endpoint dots."""
    for i, (mu, d, lo, hi) in enumerate(zip(means, depths, mins, maxs)):
        ax.plot([lo, hi], [d, d], '-', color=color, lw=1.2, zorder=3)
        ax.plot([lo, hi], [d, d], '.', color=color, ms=6, zorder=5, linestyle='none')
        ax.plot(mu, d, mean_marker, color=color, ms=7, zorder=6,
                label=label if i == 0 else None)

# ── Figure: 3 panels ──────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 8), dpi=150)

bot    = 0.44
h      = 0.52
w      = 0.272
left0  = 0.07
gap_ab = 0.015
gap_bc = 0.062

left_a = left0
left_b = left_a + w + gap_ab
left_c = left_b + w + gap_bc

ax_a = fig.add_axes([left_a, bot, w, h])
ax_b = fig.add_axes([left_b, bot, w, h], sharey=ax_a)
ax_c = fig.add_axes([left_c, bot, w, h])

# ─────────────────────────────────────────────────────────────────────────────
# (a) SR1TT vs Fischer (2018) BH1
# ─────────────────────────────────────────────────────────────────────────────
ax_a.axvline(0, color='black', lw=1.5, ls='--', zorder=1)

for (date, prof), color in zip(fischer_bh1.items(), BH1_COLORS):
    ax_a.plot(prof['temp'], prof['depth'], color=color, lw=1.8,
              marker='o', ms=4, label=f'Fischer (2018) BH1 — {date}', zorder=3)

plot_range(ax_a, sr1tt_means, sr1tt_depths, sr1tt_mins, sr1tt_maxs,
           color=SR_COLOR, label='SR1TT (2024/25, mean [min–max])')

ax_a.set_title('SR1TT [~2805 m a.s.l.]', fontsize=16, pad=8)
ax_a.set_xlabel('Temperature [°C]', fontsize=15)
ax_a.set_ylabel('Depth [m]', fontsize=15)
ax_a.set_xlim(-1.75, 0.1)
ax_a.invert_yaxis()
ax_a.set_ylim(37, 0)
ax_a.grid(True, alpha=0.3, lw=0.5)
ax_a.tick_params(labelsize=14)
ax_a.text(0.02, 0.98, '(a)', transform=ax_a.transAxes, **LABEL_KWARGS)
ax_a.text(0.03, 0.03, 'Sex Rouge (SR)', transform=ax_a.transAxes, **GLNAME_KWARGS)
ax_a.legend(**LEG_KW)

# ─────────────────────────────────────────────────────────────────────────────
# (b) SR2TT vs Fischer (2018) BH2
# ─────────────────────────────────────────────────────────────────────────────
ax_b.axvline(0, color='black', lw=1.5, ls='--', zorder=1)

for meas, color, marker in zip(fischer_bh2, BH2_COLORS, BH2_MARKERS):
    ax_b.plot(meas['temp'], meas['depth'], marker=marker, color=color,
              ms=8, ls='none',
              label=f"Fischer (2018) BH2 — {meas['date']}", zorder=3)

plot_range(ax_b, sr2tt_means, sr2tt_depths, sr2tt_mins, sr2tt_maxs,
           color=SR_COLOR, label='SR2TT (2024/25, mean [min–max])')

ax_b.set_title('SR2TT [~2777 m a.s.l.]', fontsize=16, pad=8)
ax_b.set_xlabel('Temperature [°C]', fontsize=15)
plt.setp(ax_b.get_yticklabels(), visible=False)
ax_b.set_xlim(-1.75, 0.1)
ax_b.grid(True, alpha=0.3, lw=0.5)
ax_b.tick_params(labelsize=14)
ax_b.text(0.02, 0.98, '(b)', transform=ax_b.transAxes, **LABEL_KWARGS)
ax_b.text(0.03, 0.03, 'Sex Rouge (SR)', transform=ax_b.transAxes, **GLNAME_KWARGS)
ax_b.legend(**LEG_KW)

# ─────────────────────────────────────────────────────────────────────────────
# (c) CV2TT vs Haeberli et al. (2004) — shallow sensor only (4.3 m)
#     deeper sensor excluded: failed before reaching thermal equilibrium
# ─────────────────────────────────────────────────────────────────────────────
ax_c.axvline(0, color='black', lw=1.5, ls='--', zorder=1)

for (date, prof), color in zip(haeberli_profiles.items(), HAE_COLORS):
    ax_c.plot(prof['temp'], prof['depth'], color=color, lw=1.8,
              marker='o', ms=4, label=f'Haeberli et al. (2004) — {date}', zorder=3)

plot_range(ax_c, cv1tt_means[:1], cv1tt_depths[:1], cv1tt_mins[:1], cv1tt_maxs[:1],
           color=CV_COLOR, label='CV2TT (2024/25, mean [min–max])')

ax_c.set_title('CV2TT [~3340 m a.s.l.]', fontsize=16, pad=8)
ax_c.set_xlabel('Temperature [°C]', fontsize=15)
ax_c.set_ylabel('Depth [m]', fontsize=15)
ax_c.set_xlim(-9.5, 0.5)
ax_c.invert_yaxis()
ax_c.set_ylim(14, 0)
ax_c.grid(True, alpha=0.3, lw=0.5)
ax_c.tick_params(labelsize=14)
ax_c.text(0.02, 0.98, '(c)', transform=ax_c.transAxes, **LABEL_KWARGS)
ax_c.text(0.03, 0.03, 'Corvatsch (CV)', transform=ax_c.transAxes, **GLNAME_KWARGS)
ax_c.legend(**LEG_KW)

out_path = output_dir + 'sr_cv_historical_comparison.pdf'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')


In [ ]:
# ── Compute ΔT = T_2024/25 − T_historical at sensor depths ───────────────────

# SR1TT: interpolate each BH1 profile (full) to our sensor depths
delta_sr1      = {}
delta_sr1_mins = {}
delta_sr1_maxs = {}
for date, prof in fischer_bh1.items():
    depths_h = np.array(prof['depth'])
    temps_h  = np.array(prof['temp'])
    t_interp = np.interp(np.array(sr1tt_depths), depths_h, temps_h)
    delta_sr1[date]      = np.array(sr1tt_means) - t_interp
    delta_sr1_mins[date] = np.array(sr1tt_mins)  - t_interp
    delta_sr1_maxs[date] = np.array(sr1tt_maxs)  - t_interp

# SR2TT: BH2 has only 2 isolated single-point measurements — no profile available.
# Best we can do: compare our 10 m sensor against the historical ~9-10 m readings.
# Depth mismatch is small (~0.3–0.9 m) and noted explicitly.
delta_sr2      = []
delta_sr2_mins = []
delta_sr2_maxs = []
for m in fischer_bh2:
    delta_sr2.append({
        'date':      m['date'],
        'dt':        sr2tt_means[0] - m['temp'],
        'his_depth': m['depth'],
    })
    delta_sr2_mins.append(sr2tt_mins[0] - m['temp'])
    delta_sr2_maxs.append(sr2tt_maxs[0] - m['temp'])

# CV2TT: only use shallow sensor (4.3 m) — deeper one failed before reaching
# thermal equilibrium with surrounding ice and is excluded.
cv_depths_plot = np.array(cv1tt_depths)[:1]
cv_means_plot  = np.array(cv1tt_means)[:1]
cv_mins_plot   = np.array(cv1tt_mins)[:1]
cv_maxs_plot   = np.array(cv1tt_maxs)[:1]

delta_cv1      = {}
delta_cv1_mins = {}
delta_cv1_maxs = {}
cv1_boundary_flag = []
for date, prof in haeberli_profiles.items():
    depths_h = np.array(prof['depth'])
    temps_h  = np.array(prof['temp'])
    t_interp = np.interp(cv_depths_plot, depths_h, temps_h)
    delta_cv1[date]      = cv_means_plot - t_interp
    delta_cv1_mins[date] = cv_mins_plot  - t_interp
    delta_cv1_maxs[date] = cv_maxs_plot  - t_interp
    cv1_boundary_flag.append(cv_depths_plot[0] < depths_h[0])

# ── Style ─────────────────────────────────────────────────────────────────────
SR_COLOR  = '#C0392B'
CV_COLOR  = '#C0392B'
BH1_COLORS  = ['#BBBBBB', '#888888', '#555555', '#222222']
BH2_COLORS  = ['#888888', '#444444']
BH2_MARKERS = ['s', 'D']
HAE_COLORS  = ['#DDDDDD', '#AAAAAA', '#777777', '#444444', '#111111']

DOT_KW = dict(ms=4, zorder=5, linestyle='none')  # small dots for min/max

LABEL_KWARGS = dict(
    fontsize=15, fontweight='bold', va='top', ha='left',
    bbox=dict(facecolor='white', edgecolor='none', pad=2.0)
)
GLNAME_KWARGS = dict(
    fontsize=12, fontweight='bold', va='bottom', ha='left',
    bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.4')
)
LEG_KW = dict(fontsize=10, fancybox=False, edgecolor='black',
              loc='upper left', bbox_to_anchor=(0, -0.14), borderaxespad=0)

WARM_BG  = dict(color='#FFF2F2', zorder=0)
COOL_BG  = dict(color='#F2F4FF', zorder=0)

# ── Figure layout ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 8), dpi=150)

bot    = 0.44
h      = 0.52
w      = 0.272
left0  = 0.07
gap_ab = 0.015
gap_bc = 0.022

left_a = left0
left_b = left_a + w + gap_ab
left_c = left_b + w + gap_bc

ax_a = fig.add_axes([left_a, bot, w, h])
ax_b = fig.add_axes([left_b, bot, w, h], sharey=ax_a)
ax_c = fig.add_axes([left_c, bot, w, h])

def add_bg(ax, xlim):
    ax.axvspan(xlim[0], 0,        **COOL_BG)
    ax.axvspan(0,        xlim[1], **WARM_BG)

# ── (a) SR1TT ΔT ─────────────────────────────────────────────────────────────
xlim_a = (-0.3, 1.2)
add_bg(ax_a, xlim_a)
ax_a.axvline(0, color='black', lw=1.5, ls='--', zorder=1)

for (date, dt), dmin, dmax, color in zip(
        delta_sr1.items(), delta_sr1_mins.values(), delta_sr1_maxs.values(), BH1_COLORS):
    ax_a.plot(dt,   sr1tt_depths, 'o', color=color, ms=8,  zorder=4, label=f'vs {date}')
    ax_a.plot(dmin, sr1tt_depths, '.', color=color, **DOT_KW)
    ax_a.plot(dmax, sr1tt_depths, '.', color=color, **DOT_KW)
    ax_a.plot(dt,   sr1tt_depths, color=color, lw=1.0, ls=':', alpha=0.6, zorder=3)

ax_a.set_xlim(xlim_a)
ax_a.invert_yaxis()
ax_a.set_ylim(37, 0)
ax_a.set_xlabel('ΔT [°C]  (2024/25 − historical)', fontsize=13)
ax_a.set_ylabel('Depth [m]', fontsize=13)
ax_a.set_title('SR1TT [~2805 m a.s.l.]', fontsize=14, pad=8)
ax_a.grid(True, alpha=0.3, lw=0.5)
ax_a.tick_params(labelsize=12)
ax_a.text(0.02, 0.98, '(a)', transform=ax_a.transAxes, **LABEL_KWARGS)
ax_a.text(0.03, 0.03, 'Sex Rouge (SR)', transform=ax_a.transAxes, **GLNAME_KWARGS)
ax_a.legend(**LEG_KW)

# ── (b) SR2TT ΔT (point comparison — no full profile available for BH2) ──────
xlim_b = (-0.3, 1.8)
add_bg(ax_b, xlim_b)
ax_b.axvline(0, color='black', lw=1.5, ls='--', zorder=1)

for entry, dmin, dmax, color, marker in zip(
        delta_sr2, delta_sr2_mins, delta_sr2_maxs, BH2_COLORS, BH2_MARKERS):
    depth_label = f"{entry['his_depth']:.1f} m"
    ax_b.plot(entry['dt'], entry['his_depth'], marker, color=color, ms=8, zorder=4,
              label=f"vs {entry['date']} (hist. depth {depth_label})")
    ax_b.plot(dmin, entry['his_depth'], '.', color=color, **DOT_KW)
    ax_b.plot(dmax, entry['his_depth'], '.', color=color, **DOT_KW)

ax_b.text(0.5, 0.5,
          'Only 2 historical\npoint measurements\n(depth mismatch ~0.3–0.9 m)',
          transform=ax_b.transAxes, ha='center', va='center',
          fontsize=9, color='#777777', style='italic')

ax_b.set_xlim(xlim_b)
plt.setp(ax_b.get_yticklabels(), visible=False)
ax_b.set_xlabel('ΔT [°C]  (2024/25 − historical)', fontsize=13)
ax_b.set_title('SR2TT [~2777 m a.s.l.]', fontsize=14, pad=8)
ax_b.grid(True, alpha=0.3, lw=0.5)
ax_b.tick_params(labelsize=12)
ax_b.text(0.02, 0.98, '(b)', transform=ax_b.transAxes, **LABEL_KWARGS)
ax_b.text(0.03, 0.03, 'Sex Rouge (SR)', transform=ax_b.transAxes, **GLNAME_KWARGS)
ax_b.legend(**LEG_KW)

# ── (c) CV2TT ΔT (shallow sensor only) ───────────────────────────────────────
xlim_c = (-6.0, 5.0)
add_bg(ax_c, xlim_c)
ax_c.axvline(0, color='black', lw=1.5, ls='--', zorder=1)

for (date, dt), dmin, dmax, color, extrap in zip(
        delta_cv1.items(), delta_cv1_mins.values(), delta_cv1_maxs.values(),
        HAE_COLORS, cv1_boundary_flag):
    marker = 'o' if not extrap else 's'
    ms     = 8 if not extrap else 7
    ax_c.plot(dt,   cv_depths_plot, marker, color=color, ms=ms,  zorder=4,
              label=f'vs {date}' + (' *' if extrap else ''))
    ax_c.plot(dmin, cv_depths_plot, '.', color=color, **DOT_KW)
    ax_c.plot(dmax, cv_depths_plot, '.', color=color, **DOT_KW)
    ax_c.plot(dt,   cv_depths_plot, color=color, lw=1.0, ls=':', alpha=0.6, zorder=3)

ax_c.set_xlim(xlim_c)
ax_c.invert_yaxis()
ax_c.set_ylim(14, 0)
ax_c.set_xlabel('ΔT [°C]  (2024/25 − historical)', fontsize=13)
ax_c.set_ylabel('Depth [m]', fontsize=13)
ax_c.set_title('CV2TT [~3340 m a.s.l.]', fontsize=14, pad=8)
ax_c.grid(True, alpha=0.3, lw=0.5)
ax_c.tick_params(labelsize=12)
ax_c.text(0.02, 0.98, '(c)', transform=ax_c.transAxes, **LABEL_KWARGS)
ax_c.text(0.03, 0.03, 'Corvatsch (CV)', transform=ax_c.transAxes, **GLNAME_KWARGS)
ax_c.text(1.0, -0.01,
          '* shallowest depth extrapolated beyond hist. profile range',
          transform=ax_c.transAxes, ha='right', va='top',
          fontsize=8, color='#777777', style='italic')
ax_c.legend(**LEG_KW)

# Shared annotation: direction labels
for ax in [ax_a, ax_b, ax_c]:
    ax.text(0.97, 0.5, '→ warmer', transform=ax.transAxes,
            ha='right', va='center', fontsize=9, color='#AA3333', alpha=0.7, style='italic')
    ax.text(0.03, 0.5, '← cooler', transform=ax.transAxes,
            ha='left', va='center', fontsize=9, color='#3333AA', alpha=0.7, style='italic')

out_path = output_dir + 'sr_cv_historical_delta_T.pdf'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')


In [ ]:
# Load CV2TT (logger 12, CT2TT file, depths: white=4.3m, black=9.3m at installation)
depth_CV2TT_path = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_cv2tt.csv")
CV2TT_hist = ThermistorData(TT_icetemp_dir + "CT2TT_20240828_20250905_spliced.csv", ",", depth_CV2TT_path)
cv2tt_ts_hist = CV2TT_hist.get_ntc_data_with_offsets('12', corrected_offsets_TT, aggregate=None)

cv2tt_white_mean = cv2tt_ts_hist['White Probe Temperature'].mean()
cv2tt_white_min  = cv2tt_ts_hist['White Probe Temperature'].min()
cv2tt_white_max  = cv2tt_ts_hist['White Probe Temperature'].max()
cv2tt_black_mean = cv2tt_ts_hist['Black Probe Temperature'].mean()
cv2tt_black_min  = cv2tt_ts_hist['Black Probe Temperature'].min()
cv2tt_black_max  = cv2tt_ts_hist['Black Probe Temperature'].max()
n_black = cv2tt_ts_hist['Black Probe Temperature'].notna().sum()

print(f"CV2TT white (4.3 m): mean={cv2tt_white_mean:.3f}, min={cv2tt_white_min:.3f}, max={cv2tt_white_max:.3f}")
print(f"CV2TT black (9.3 m): mean={cv2tt_black_mean:.3f}, min={cv2tt_black_min:.3f}, max={cv2tt_black_max:.3f}  (n={n_black}, ~17 h only)")


In [ ]:
"""
Like-for-like seasonal comparison: extract our measured temperature on the same
day-of-year as each historical snapshot, then compute Δ = ours − historical.

SR1TT (logger #1)  vs  Fischer (2018) BH1  — dates: Nov 2013, Sep 2014, Jul 2015, Sep 2015
SR2TT (logger #2)  vs  Fischer (2018) BH2  — dates: Sep 2014, Jul 2015
CV2TT (logger #12) vs  Haeberli et al. (2004) — dates: Sep 1999 – Sep 2000

Strategy: for each historical date, find the same calendar day/month in our
2024/25 record and read the daily mean at that point.
"""
from scipy.interpolate import interp1d

# ── load full time series with offset corrections ────────────────────────────
sr1tt_ts = SR1TT.get_ntc_data_with_offsets('1',  corrected_offsets_TT, aggregate=None)
sr2tt_ts = SR2TT.get_ntc_data_with_offsets('2',  corrected_offsets_TT, aggregate=None)
cv2tt_ts = CV2TT.get_ntc_data_with_offsets('12', corrected_offsets_TT, aggregate=None)

# resample to daily means for stable lookup
def daily_mean(ts):
    ts = ts.copy()
    ts['TIME'] = pd.to_datetime(ts['TIME'])
    ts = ts.set_index('TIME')
    return ts.resample('D').mean()

sr1_daily = daily_mean(sr1tt_ts)
sr2_daily = daily_mean(sr2tt_ts)
cv2_daily = daily_mean(cv2tt_ts)

def get_same_doy(daily_df, hist_date_str, probe):
    """Return temperature for our record at the same day-of-year as hist_date_str.
    
    Looks for (month, day) in 2024 first, then 2025.
    hist_date_str: e.g. '11 Sep 2014'
    probe: 'White Probe Temperature' or 'Black Probe Temperature'
    """
    hist_dt = pd.to_datetime(hist_date_str, format='%d %b %Y')
    for year in [2024, 2025]:
        candidate = pd.Timestamp(year=year, month=hist_dt.month, day=hist_dt.day)
        if candidate in daily_df.index:
            val = daily_df.loc[candidate, probe]
            if not np.isnan(val):
                return val, candidate
    return np.nan, None

def interp_hist(depths_hist, temps_hist, query_depths):
    f = interp1d(depths_hist, temps_hist, bounds_error=False, fill_value=np.nan)
    return [float(f(d)) for d in query_depths]

# probe assignment: white = shallow, black = deep
# SR1TT: white=10.0 m, black=15.2 m
# SR2TT: white=10.0 m, black=14.4 m
# CV2TT: white=4.3 m,  black=9.3 m

print("=" * 65)
print("SR1TT  vs  Fischer (2018) BH1  (~2805 m a.s.l.)")
print(f"  Depths: white=10.0 m,  black=15.2 m")
print()
sr1_deltas = {10.0: [], 15.2: []}
for date, prof in fischer_bh1.items():
    hist_at_obs = interp_hist(prof["depth"], prof["temp"], sr1tt_depths)
    obs_w, t_w = get_same_doy(sr1_daily, date, 'White Probe Temperature')
    obs_b, t_b = get_same_doy(sr1_daily, date, 'Black Probe Temperature')
    obs = [obs_w, obs_b]
    print(f"  {date}  (matched to {t_w.date() if t_w else 'n/a'} / {t_b.date() if t_b else 'n/a'})")
    for depth, o, hist in zip(sr1tt_depths, obs, hist_at_obs):
        if np.isnan(hist) or np.isnan(o):
            print(f"    {depth:.1f} m: n/a")
        else:
            d = o - hist
            sr1_deltas[depth].append(d)
            print(f"    {depth:.1f} m: hist={hist:+.3f}  ours={o:+.3f}  Δ={d:+.3f} °C")
print()
print("  --- Mean Δ (same-day-of-year comparison) ---")
for depth in sr1tt_depths:
    ds = sr1_deltas[depth]
    if ds:
        print(f"  {depth:.1f} m:  mean Δ = {np.mean(ds):+.3f} °C  "
              f"(range {min(ds):+.3f} to {max(ds):+.3f},  n={len(ds)})")

print()
print("=" * 65)
print("SR2TT  vs  Fischer (2018) BH2  (~2777 m a.s.l.)")
print(f"  Depths: white=10.0 m,  black=14.4 m")
print()
sr2_deltas = []
for meas in fischer_bh2:
    obs_w, t_w = get_same_doy(sr2_daily, meas['date'], 'White Probe Temperature')
    print(f"  {meas['date']}  (matched to {t_w.date() if t_w else 'n/a'})")
    if np.isnan(obs_w):
        print(f"    10.0 m: n/a")
    else:
        d = obs_w - meas['temp']
        sr2_deltas.append(d)
        print(f"    hist@{meas['depth']:.2f} m={meas['temp']:+.3f}  ours@10.0 m={obs_w:+.3f}  Δ={d:+.3f} °C")
print()
print("  --- Mean Δ (same-day-of-year comparison) ---")
if sr2_deltas:
    print(f"  10.0 m:  mean Δ = {np.mean(sr2_deltas):+.3f} °C  "
          f"(range {min(sr2_deltas):+.3f} to {max(sr2_deltas):+.3f},  n={len(sr2_deltas)})")

print()
print("=" * 65)
print("CV2TT  vs  Haeberli et al. (2004)  (~3340 m a.s.l.)")
print(f"  Depths: white=4.3 m,  black=9.3 m  (black: ~17 h only)")
print()
cv2_deltas = {4.3: [], 9.3: []}
for date, prof in haeberli_profiles.items():
    hist_at_obs = interp_hist(prof["depth"], prof["temp"], cv1tt_depths)
    obs_w, t_w = get_same_doy(cv2_daily, date, 'White Probe Temperature')
    obs_b, t_b = get_same_doy(cv2_daily, date, 'Black Probe Temperature')
    obs = [obs_w, obs_b]
    print(f"  {date}  (matched to {t_w.date() if t_w else 'n/a'} / {t_b.date() if t_b else 'n/a'})")
    for depth, o, hist in zip(cv1tt_depths, obs, hist_at_obs):
        if np.isnan(hist) or np.isnan(o):
            print(f"    {depth:.1f} m: n/a")
        else:
            d = o - hist
            cv2_deltas[depth].append(d)
            print(f"    {depth:.1f} m: hist={hist:+.3f}  ours={o:+.3f}  Δ={d:+.3f} °C")
print()
print("  --- Mean Δ (same-day-of-year comparison) ---")
for depth in cv1tt_depths:
    ds = cv2_deltas[depth]
    if ds:
        print(f"  {depth:.1f} m:  mean Δ = {np.mean(ds):+.3f} °C  "
              f"(range {min(ds):+.3f} to {max(ds):+.3f},  n={len(ds)})")
    else:
        print(f"  {depth:.1f} m:  no valid matches")


In [ ]:
"""
Option 2 (revised v8): top-row whiskers with dot endpoints,
restored lower-panel warmer/cooler labels, lower-panel backgrounds sent behind all content,
panel d uses vertical summary lines only.
"""
from scipy.interpolate import interp1d


SR_COLOR = "#C0392B"
BH1_COLORS = ["#BBBBBB", "#888888", "#555555", "#222222"]
BH2_COLORS = ["#888888", "#444444"]
BH2_MARKERS = ["s", "D"]
HAE_COLORS = ["#DDDDDD", "#AAAAAA", "#777777", "#444444", "#111111"]
LABEL_KWARGS = dict(fontsize=16, fontweight="bold", va="top", ha="left",
                    bbox=dict(facecolor="white", edgecolor="none", pad=2.0))
GLNAME_KWARGS = dict(fontsize=15, fontweight="bold", va="bottom", ha="left",
                     bbox=dict(facecolor="white", edgecolor="black", boxstyle="round,pad=0.3"))
LEG_KW = dict(fontsize=13, fancybox=False, edgecolor="black", framealpha=0.95,
             handlelength=1.4, handletextpad=0.4, labelspacing=0.3, borderpad=0.4)
LEG_Y = -0.33

fischer_bh1 = {
    '17 Nov 2013': {
        'depth': [1.9, 3.9, 4.9, 5.9, 7.9, 9.9, 14.9, 19.9, 24.9, 34.9],
        'temp':  [-0.51, -0.72, -0.82, -0.90, -0.95, -0.88, -0.52, -0.12, -0.05, -0.05],
    },
    '11 Sep 2014': {
        'depth': [1.5, 3.5, 4.5, 5.5, 7.5, 9.5, 14.5, 19.5, 24.5, 34.5],
        'temp':  [-0.38, -0.88, -1.00, -1.08, -1.07, -0.95, -0.55, -0.17, -0.06, -0.06],
    },
    '12 Jul 2015': {
        'depth': [1.0, 3.0, 4.0, 5.0, 7.0, 9.0, 14.0, 19.0, 24.0, 34.0],
        'temp':  [-0.53, -1.25, -1.45, -1.53, -1.38, -1.26, -0.54, -0.15, 0.02, -0.03],
    },
    '21 Sep 2015': {
        'depth': [0.6, 1.6, 2.6, 4.6, 6.6, 11.6, 16.6, 21.6, 31.6],
        'temp':  [-0.90, -0.39, -0.67, -0.96, -1.12, -0.60, -0.18, 0.02, -0.03],
    },
}

fischer_bh2 = [
    {'date': '11 Sep 2014', 'depth': 9.70, 'temp': -1.27},
    {'date': '12 Jul 2015', 'depth': 9.05, 'temp': -1.38},
]

haeberli_profiles = {
    '14 Sep 1999': {
        'depth': [2.485, 3.040, 3.310, 3.562, 3.831, 4.387, 4.909, 5.464, 5.986,
                  6.576, 7.115, 7.654, 8.479, 9.052, 9.860, 10.433, 10.938, 11.528,
                  12.017, 12.640, 12.910],
        'temp':  [-0.059, -0.876, -1.081, -1.353, -1.660, -2.137, -2.580, -2.956,
                  -3.195, -3.571, -3.844, -4.050, -4.290, -4.360, -4.499, -4.501,
                  -4.537, -4.471, -4.438, -4.373, -4.374],
    },
    '1 Dec 1999': {
        'depth': [1.026, 1.397, 1.834, 2.171, 2.760, 3.382, 3.887, 4.358, 5.014,
                  5.923, 6.495, 7.084, 7.976, 9.423, 11.139, 12.216, 12.957],
        'temp':  [-5.322, -5.186, -5.153, -4.847, -4.203, -3.559, -3.186, -3.051,
                  -2.881, -2.881, -2.847, -2.949, -3.220, -3.492, -3.831, -3.864, -3.831],
    },
    '5 Apr 2000': {
        'depth': [0.976, 2.204, 3.248, 5.486, 7.252, 8.716, 10.382, 12.065, 12.974],
        'temp':  [-8.068, -7.898, -7.356, -6.169, -5.085, -4.542, -4.203, -3.864, -3.763],
    },
    '19 Jun 2000': {
        'depth': [0.993, 2.036, 3.079, 4.005, 4.998, 6.832, 7.976, 8.986, 10.046,
                  11.661, 13.024],
        'temp':  [-0.373, -0.508, -2.712, -4.169, -4.983, -5.390, -5.288, -5.017,
                  -4.983, -4.576, -4.305],
    },
    '20 Sep 2000': {
        'depth': [1.010, 1.565, 2.188, 2.726, 3.786, 4.930, 5.822, 6.916, 8.161,
                  8.868, 10.433, 12.115, 12.974],
        'temp':  [-1.390, -1.119, -0.983, -1.051, -1.763, -2.983, -3.356, -3.492,
                  -3.695, -3.966, -4.339, -4.576, -4.271],
    },
}

def daily_mean(ts):
    ts = ts.copy()
    ts["TIME"] = pd.to_datetime(ts["TIME"])
    return ts.set_index("TIME").resample("D").mean()


def doy_temp(daily_df, hist_date_str, probe):
    hist_dt = pd.to_datetime(hist_date_str, format="%d %b %Y")
    for year in (2024, 2025):
        candidate = pd.Timestamp(year=year, month=hist_dt.month, day=hist_dt.day)
        if candidate in daily_df.index:
            val = daily_df.loc[candidate, probe]
            if not np.isnan(val):
                return float(val)
    return np.nan


def interp_hist(depths_h, temps_h, query_depth):
    f = interp1d(depths_h, temps_h, bounds_error=False, fill_value=np.nan)
    return float(f(query_depth))


def add_bg(ax, xlim):
    ax.axvspan(xlim[0], 0, color="#F2F4FF", zorder=-20)
    ax.axvspan(0, xlim[1], color="#FFF2F2", zorder=-20)


def plot_summary_with_vertical_line(ax, mean_val, color, line_color=None):
    lc = color if line_color is None else line_color
    ax.axvline(mean_val, color=lc, lw=1.4, alpha=0.9, zorder=25)


# Time series with offsets
sr1tt_ts = SR1TT.get_ntc_data_with_offsets('1', corrected_offsets_TT, aggregate=None)
sr2tt_ts = SR2TT.get_ntc_data_with_offsets('2', corrected_offsets_TT, aggregate=None)
cv2tt_ts = CV2TT.get_ntc_data_with_offsets('12', corrected_offsets_TT, aggregate=None)

sr1_daily = daily_mean(sr1tt_ts)
sr2_daily = daily_mean(sr2tt_ts)
cv2_daily = daily_mean(cv2tt_ts)

# Figure layout
FIG_W = 14.0
LEFT, RIGHT = 0.07, 0.97
G_SMALL = 0.02
W = 0.88 / 3.35
G_LARGE = 0.28 * W
A_L = LEFT
B_L = A_L + W + G_SMALL
C_L = B_L + W + G_LARGE
WSPACE_AB = G_SMALL / W
panel_w_in = W * FIG_W
TOP_H = panel_w_in
BOT_H = panel_w_in / 2
GAP = 0.80
LEG_SP = 2.0
TOP_M = 0.35
FIG_H = TOP_M + TOP_H + GAP + BOT_H + LEG_SP
fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=150)

def to_norm(in_from_bottom):
    return in_from_bottom / FIG_H

top_top = to_norm(FIG_H - TOP_M)
top_bot = to_norm(FIG_H - TOP_M - TOP_H)
bot_top = to_norm(LEG_SP + BOT_H)
bot_bot = to_norm(LEG_SP)

gs_AB_top = fig.add_gridspec(1, 2, left=A_L, right=B_L + W, top=top_top, bottom=top_bot,
                              wspace=WSPACE_AB, width_ratios=[1, 1])
gs_C_top = fig.add_gridspec(1, 1, left=C_L, right=RIGHT, top=top_top, bottom=top_bot)
gs_AB_bot = fig.add_gridspec(1, 2, left=A_L, right=B_L + W, top=bot_top, bottom=bot_bot,
                              wspace=WSPACE_AB, width_ratios=[1, 1])
gs_C_bot = fig.add_gridspec(1, 1, left=C_L, right=RIGHT, top=bot_top, bottom=bot_bot)

ax_a = fig.add_subplot(gs_AB_top[0])
ax_b = fig.add_subplot(gs_AB_top[1], sharey=ax_a)
ax_c = fig.add_subplot(gs_C_top[0])
ax_d = fig.add_subplot(gs_AB_bot[0], sharey=ax_a)
ax_e = fig.add_subplot(gs_AB_bot[1], sharey=ax_a)
ax_f = fig.add_subplot(gs_C_bot[0], sharey=ax_c)

TEMP_XLIM = (-1.7, 0.1)
TEMP_XTICKS = [-1.5, -1.0, -0.5, 0.0]

# ── GLAMOS helpers: surface ablation → historical sensor depth ─────────────────────────
ICE_DENSITY = 900.0  # kg m⁻³

def _mb_at_date(csv_path, query_date):
    """Interpolate cumulative mass balance (mm w.e.) at query_date from GLAMOS CSV."""
    df = pd.read_csv(csv_path, skiprows=2, header=0,
                     names=['Glacier', 'ID', 'Start', 'End', 'Value'])
    df['Start'] = pd.to_datetime(df['Start'])
    df['End']   = pd.to_datetime(df['End'])
    df['Value'] = pd.to_numeric(df['Value'])
    qdt = pd.to_datetime(query_date)
    if qdt <= df['Start'].iloc[0]:
        return 0.0
    if qdt >= df['End'].iloc[-1]:
        return float(df['Value'].iloc[-1])
    for i in range(len(df)):
        row = df.iloc[i]
        start_mb = float(df.iloc[i - 1]['Value']) if i > 0 else 0.0
        if row['Start'] <= qdt <= row['End']:
            span = max((row['End'] - row['Start']).days, 1)
            frac = (qdt - row['Start']).days / span
            return start_mb + frac * (float(row['Value']) - start_mb)
    return float(df['Value'].iloc[-1])

def _surface_abl_ice(mb_hist_mmwe, mb_now_mmwe):
    """Cumulative MB difference (mm w.e.) → surface ablation in metres of ice."""
    return abs(mb_now_mmwe - mb_hist_mmwe) / 1000.0 * (1000.0 / ICE_DENSITY)

SR_MB_CSV    = os.path.join(project_root, 'data', 'glamos', 'mass_balance_cumulative_sexrouge.csv')
INSTALL_DATE = '2024-08-01'
_mb_sr_now   = _mb_at_date(SR_MB_CSV, INSTALL_DATE)

CV_MB_CSV    = os.path.join(project_root, 'data', 'glamos', 'mass_balance_cumulative_corvatsch.csv')
_mb_cv_now   = _mb_at_date(CV_MB_CSV, INSTALL_DATE)

# Mean surface ablation from each Fischer measurement date to sensor installation
_sr_mean_abl = float(np.mean([_surface_abl_ice(_mb_at_date(SR_MB_CSV, d), _mb_sr_now)
                               for d in fischer_bh1]))
# Corvatsch GLAMOS starts 2012 (predates Haeberli 1999-2000); use full available record as min estimate
_cv_abl = _surface_abl_ice(0.0, _mb_cv_now)

def _shifted_diamond(ax, temp_mean, temp_min, temp_max, current_depth, ablation_m,
                      edge_color=SR_COLOR, add_label=False, label_str=None, zorder=3.8):
    """Hollow (white fill, colored edge) diamond: our measured temperature at the historical
    depth, i.e. current depth + surface ablation since the historical measurement period."""
    hist_d = current_depth + ablation_m
    ax.errorbar(temp_mean, hist_d,
                xerr=[[temp_mean - temp_min], [temp_max - temp_mean]],
                fmt='D', color='white', markeredgecolor=edge_color, markeredgewidth=1.5,
                ecolor=edge_color, elinewidth=1.5, capsize=0,
                ms=7, zorder=zorder, label=label_str if add_label else '_nolegend_')
    ax.plot([temp_min, temp_max], [hist_d, hist_d],
            linestyle='none', marker='o', color=edge_color, ms=4, zorder=zorder)

# (a) SR1TT
ax_a.axvline(0, color="black", lw=1.5, ls="--", zorder=1)
for (date, prof), color in zip(fischer_bh1.items(), BH1_COLORS):
    ax_a.plot(prof["temp"], prof["depth"], color=color, lw=1.8, marker="o", ms=4,
              label=f"Fischer (2018) BH1 — {date}", zorder=3)
ax_a.errorbar([ -0.397, -0.273 ], [10.0, 15.2],
              xerr=[[0.269, 0.130], [0.397, 0.079]],
              fmt="D", color=SR_COLOR, ecolor=SR_COLOR, elinewidth=1.5, capsize=0,
              ms=7, label="SR1TT (mean [min–max], 2024/25)", zorder=4)
ax_a.plot([-0.397 - 0.269, -0.397 + 0.397], [10.0, 10.0], linestyle="none", marker="o", color=SR_COLOR, ms=4, zorder=5)
ax_a.plot([-0.273 - 0.130, -0.273 + 0.079], [15.2, 15.2], linestyle="none", marker="o", color=SR_COLOR, ms=4, zorder=5)
# Hollow diamonds: our temperatures shifted to historical depths
_shifted_diamond(ax_a, -0.397, -0.397 - 0.269, -0.397 + 0.397, 10.0, _sr_mean_abl,
                 SR_COLOR, add_label=True, label_str='SR1TT at Fischer (2018) era depth')
_shifted_diamond(ax_a, -0.273, -0.273 - 0.130, -0.273 + 0.079, 15.2, _sr_mean_abl, SR_COLOR)
ax_a.invert_yaxis(); ax_a.set_ylim(37, 0)
ax_a.set_xlim(TEMP_XLIM); ax_a.set_xticks(TEMP_XTICKS)
ax_a.set_xlabel("Temperature [°C]", fontsize=14)
ax_a.set_ylabel("Depth [m]", fontsize=14)
ax_a.set_title("SR1TT [≈2805 m a.s.l.]", fontsize=15, pad=6)
ax_a.grid(True, alpha=0.3, lw=0.5); ax_a.tick_params(labelsize=13)
ax_a.text(0.02, 0.98, "(a)", transform=ax_a.transAxes, **LABEL_KWARGS)
ax_a.text(0.03, 0.03, "Sex Rouge (SR)", transform=ax_a.transAxes, **GLNAME_KWARGS)

# (b) SR2TT
ax_b.axvline(0, color="black", lw=1.5, ls="--", zorder=1)
for meas, color, marker in zip(fischer_bh2, BH2_COLORS, BH2_MARKERS):
    ax_b.plot(meas["temp"], meas["depth"], marker, color=color, ms=8, ls="none",
              label=f"Fischer (2018) BH2 — {meas['date']}", zorder=3)
for i, (mean_val, depth_val, min_val, max_val) in enumerate(zip(sr2tt_means, sr2tt_depths, sr2tt_mins, sr2tt_maxs)):
    ax_b.errorbar(mean_val, depth_val, xerr=[[mean_val - min_val], [max_val - mean_val]],
                  fmt="D", color=SR_COLOR, ecolor=SR_COLOR, elinewidth=1.5, capsize=0,
                  ms=7, label="SR2TT (mean [min–max], 2024/25)" if i == 0 else None, zorder=4)
    ax_b.plot([min_val, max_val], [depth_val, depth_val], linestyle="none", marker="o", color=SR_COLOR, ms=4, zorder=5)
for i, (mean_val, depth_val, min_val, max_val) in enumerate(zip(sr2tt_means, sr2tt_depths, sr2tt_mins, sr2tt_maxs)):
    _shifted_diamond(ax_b, mean_val, min_val, max_val, depth_val, _sr_mean_abl,
                     SR_COLOR, add_label=(i == 0), label_str='SR2TT at Fischer (2018) era depth')

ax_b.set_xlim(TEMP_XLIM); ax_b.set_xticks(TEMP_XTICKS)
ax_b.set_xlabel("Temperature [°C]", fontsize=14)
ax_b.set_title("SR2TT [≈2777 m a.s.l.]", fontsize=15, pad=6)
ax_b.grid(True, alpha=0.3, lw=0.5)
ax_b.tick_params(labelsize=13, labelleft=False)
ax_b.text(0.02, 0.98, "(b)", transform=ax_b.transAxes, **LABEL_KWARGS)
ax_b.text(0.03, 0.03, "Sex Rouge (SR)", transform=ax_b.transAxes, **GLNAME_KWARGS)

# (c) CV2TT
ax_c.axvline(0, color="black", lw=1.5, ls="--", zorder=1)
for (date, prof), color in zip(haeberli_profiles.items(), HAE_COLORS):
    ax_c.plot(prof["temp"], prof["depth"], color=color, lw=1.8, marker="o", ms=4,
              label=f"Haeberli et al. (2004) — {date}", zorder=3)
cv_mean = float(cv2tt_white_mean)
cv_min = float(cv2tt_white_min)
cv_max = float(cv2tt_white_max)
ax_c.errorbar([cv_mean], [4.3],
              xerr=[[cv_mean - cv_min], [cv_max - cv_mean]],
              fmt="D", color=SR_COLOR, ecolor=SR_COLOR, elinewidth=1.5, capsize=0,
              ms=7, label="CV2TT (mean [min–max], 2024/25)", zorder=4)
ax_c.plot([cv_min, cv_max], [4.3, 4.3], linestyle="none", marker="o", color=SR_COLOR, ms=4, zorder=5)
_shifted_diamond(ax_c, cv_mean, cv_min, cv_max, 4.3, _cv_abl,
                 SR_COLOR, add_label=True, label_str='CV2TT at Haeberli et al. (2004) era depth')

ax_c.invert_yaxis(); ax_c.set_ylim(max(14, 4.3 + _cv_abl + 2), 0)
ax_c.set_xlabel("Temperature [°C]", fontsize=14)
ax_c.set_ylabel("Depth [m]", fontsize=14)
ax_c.set_title("CV2TT [≈3340 m a.s.l.]", fontsize=15, pad=6)
ax_c.grid(True, alpha=0.3, lw=0.5); ax_c.tick_params(labelsize=13)
ax_c.text(0.02, 0.98, "(c)", transform=ax_c.transAxes, **LABEL_KWARGS)
ax_c.text(0.03, 0.03, "Corvatsch (CV)", transform=ax_c.transAxes, **GLNAME_KWARGS)

# (d) SR1TT delta-T: separate 10 m and 15.2 m summaries
xlim_d = (-1.0, 1.0)
xticks_d = [-1.0, -0.5, 0.0, 0.5, 1.0]
add_bg(ax_d, xlim_d)
ax_d.axvline(0, color="black", lw=1.5, ls="--", zorder=1)
sr1_specs = [
    {"depth": 10.0, "probe": "White Probe Temperature", "marker": "o", "color": "#8A8A8A", "line_color": "#222222", "text_color": "#222222", "label": "10.0 m"},
    {"depth": 15.2, "probe": "Black Probe Temperature", "marker": "s", "color": "#222222", "line_color": "#222222", "text_color": "#222222", "label": "15.2 m"},
]
for spec in sr1_specs:
    values = []
    for date, prof in fischer_bh1.items():
        hist = interp_hist(prof["depth"], prof["temp"], spec["depth"])
        obs = doy_temp(sr1_daily, date, spec["probe"])
        if not (np.isnan(hist) or np.isnan(obs)):
            delta = obs - hist
            values.append(delta)
            ax_d.plot(delta, spec["depth"], "x", color="#555555", ms=8, mew=2.0, alpha=0.85, zorder=4)
    mean_val = float(np.mean(values))
    text_color = "#AA3333" if mean_val >= 0 else "#3333AA"
    ax_d.axvline(mean_val, color=text_color, lw=1.4, alpha=0.9, zorder=25)
    ann_y = 35.0
    ann_x = mean_val + 0.05 if spec["depth"] == 10.0 else mean_val - 0.05
    ann_ha = "left" if spec["depth"] == 10.0 else "right"
    ax_d.text(ann_x, ann_y, f'$\\overline{{\\Delta T}}$ ({spec["depth"]:.1f} m) = {mean_val:+.2f}°C',
              color=text_color, fontsize=11, ha=ann_ha, va="bottom",
              bbox=dict(facecolor="white", edgecolor="none", alpha=1.0, pad=0.15),
              zorder=30, clip_on=False)
ax_d.set_xlim(xlim_d)
ax_d.set_xticks(xticks_d)
ax_d.set_xlabel("ΔT [°C]  (2024/25 − historical)", fontsize=14)
ax_d.set_ylabel("Depth [m]", fontsize=14)
ax_d.grid(True, alpha=0.3, lw=0.5); ax_d.tick_params(labelsize=13)
ax_d.set_title("Same-day-of-year $\\Delta T$", fontsize=13, color="black", pad=4)
ax_d.text(0.02, 0.98, "(d)", transform=ax_d.transAxes, **LABEL_KWARGS)
ax_d.text(0.97, 0.5, "→ warmer", transform=ax_d.transAxes, ha="right", va="center",
          fontsize=12, color="#AA3333", alpha=0.7, style="italic", zorder=30, clip_on=False)
ax_d.text(0.03, 0.5, "← cooler", transform=ax_d.transAxes, ha="left", va="center",
          fontsize=12, color="#3333AA", alpha=0.7, style="italic", zorder=30, clip_on=False)

# (e) SR2TT delta-T: one shallow-depth summary
xlim_e = (-1.0, 1.0)
xticks_e = [-1.0, -0.5, 0.0, 0.5, 1.0]
add_bg(ax_e, xlim_e)
ax_e.axvline(0, color="black", lw=1.5, ls="--", zorder=1)
sr2_values = []
for meas, color, marker in zip(fischer_bh2, BH2_COLORS, BH2_MARKERS):
    obs = doy_temp(sr2_daily, meas["date"], "White Probe Temperature")
    if np.isnan(obs):
        continue
    delta = obs - meas["temp"]
    sr2_values.append(delta)
    ax_e.plot(delta, meas["depth"], "x", color="#555555", ms=8, mew=2.0, zorder=4)
mean_val = float(np.mean(sr2_values))
_mc_e = "#AA3333" if mean_val >= 0 else "#3333AA"
ax_e.axvline(mean_val, color=_mc_e, lw=1.4, alpha=0.85, zorder=5)
ax_e.text(mean_val - 0.05, 35.0, f'$\\overline{{\\Delta T}}$ ({sr2tt_depths[0]:.1f} m) = {mean_val:+.2f}°C', color=_mc_e, fontsize=11,
          ha="right", va="bottom", bbox=dict(facecolor="white", edgecolor="none", alpha=1.0, pad=0.15),
          zorder=30, clip_on=False)
ax_e.set_xlim(xlim_e)
ax_e.set_xlabel("ΔT [°C]  (2024/25 − historical)", fontsize=14)
ax_e.set_xticks(xticks_e)
ax_e.grid(True, alpha=0.3, lw=0.5)
ax_e.tick_params(labelsize=13, labelleft=False)
ax_e.set_title("Same-day-of-year $\\Delta T$", fontsize=13, color="black", pad=4)
ax_e.text(0.02, 0.98, "(e)", transform=ax_e.transAxes, **LABEL_KWARGS)
ax_e.text(0.97, 0.5, "→ warmer", transform=ax_e.transAxes, ha="right", va="center",
          fontsize=12, color="#AA3333", alpha=0.7, style="italic", zorder=30, clip_on=False)
ax_e.text(0.03, 0.5, "← cooler", transform=ax_e.transAxes, ha="left", va="center",
          fontsize=12, color="#3333AA", alpha=0.7, style="italic", zorder=30, clip_on=False)

# (f) CV2TT delta-T: shallow sensor only
xlim_f = (-1.5, 1.5)
xticks_f = [-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5]
add_bg(ax_f, xlim_f)
ax_f.axvline(0, color="black", lw=1.5, ls="--", zorder=1)
cv_values = []
for date, prof in haeberli_profiles.items():
    obs = doy_temp(cv2_daily, date, "White Probe Temperature")
    hist = interp_hist(prof["depth"], prof["temp"], 4.3)
    if not (np.isnan(obs) or np.isnan(hist)):
        delta = obs - hist
        cv_values.append(delta)
        ax_f.plot(delta, 4.3, "x", color="#555555", ms=8, mew=2.0, zorder=4)
mean_val = float(np.mean(cv_values))
_mc_f = "#AA3333" if mean_val >= 0 else "#3333AA"
ax_f.axvline(mean_val, color=_mc_f, lw=1.4, alpha=0.85, zorder=5)
ax_f.text(mean_val - 0.05, max(14, 4.3 + _cv_abl + 2) - 2, f'$\\overline{{\\Delta T}}$ ({cv1tt_depths[0]:.1f} m) = {mean_val:+.2f}°C', color=_mc_f, fontsize=11,
          ha="right", va="bottom", bbox=dict(facecolor="white", edgecolor="none", alpha=1.0, pad=0.15),
          zorder=30, clip_on=False)
ax_f.set_xlim(xlim_f)
ax_f.set_xticks(xticks_f)
ax_f.set_xlabel("ΔT [°C]  (2024/25 − historical)", fontsize=14)
ax_f.set_ylabel("Depth [m]", fontsize=14)
ax_f.grid(True, alpha=0.3, lw=0.5)
ax_f.tick_params(labelsize=13)
ax_f.set_title("Same-day-of-year $\\Delta T$", fontsize=13, color="black", pad=4)
ax_f.text(0.02, 0.98, "(f)", transform=ax_f.transAxes, **LABEL_KWARGS)
ax_f.text(0.97, 0.5, "→ warmer", transform=ax_f.transAxes, ha="right", va="center",
          fontsize=12, color="#AA3333", alpha=0.7, style="italic", zorder=30, clip_on=False)
ax_f.text(0.03, 0.5, "← cooler", transform=ax_f.transAxes, ha="left", va="center",
          fontsize=12, color="#3333AA", alpha=0.7, style="italic", zorder=30, clip_on=False)

# Combined legends placed slightly lower
from matplotlib.lines import Line2D
_x_handle = Line2D([0], [0], marker="x", color="#555555", ms=8, mew=2.0,
                   ls="none", label="Same-day-of-year $\\Delta T$")
h_a, l_a = ax_a.get_legend_handles_labels()
h_a.append(_x_handle); l_a.append(_x_handle.get_label())
ax_d.legend(handles=h_a, labels=l_a, loc="upper left", bbox_to_anchor=(0.0, LEG_Y),
            borderaxespad=0, **LEG_KW)
h_b, l_b = ax_b.get_legend_handles_labels()
h_b.append(_x_handle); l_b.append(_x_handle.get_label())
ax_e.legend(handles=h_b, labels=l_b, loc="upper left", bbox_to_anchor=(0.0, LEG_Y),
            borderaxespad=0, **LEG_KW)
h_c, l_c = ax_c.get_legend_handles_labels()
h_c.append(_x_handle); l_c.append(_x_handle.get_label())
ax_f.legend(handles=h_c, labels=l_c, loc="upper left", bbox_to_anchor=(0.0, LEG_Y),
            borderaxespad=0, **LEG_KW)

plt.savefig(os.path.join(project_root, "figures", "paper", "fig04_sr_cv_historical_comparison_dT.pdf"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved: sr_cv_historical_comparison_dT.pdf")

In [ ]:
# Temperature gradient and extrapolated temperate-ice depth for SR1TT and SR2TT

datasets = {
    "SR1TT": {"depths": [10.0, 15.2], "means": [-0.397, -0.273]},
    "SR2TT": {"depths": [10.0, 14.4], "means": [-0.348, -0.300]},
}

print(f"{'Site':<8} {'Gradient (°C/m)':<18} {'Extrap. depth to 0°C (m)':<28} {'Notes'}")
print("-" * 75)
for site, d in datasets.items():
    z = np.array(d["depths"])
    T = np.array(d["means"])

    # Linear gradient (positive = warming with depth)
    dT_dz = (T[1] - T[0]) / (z[1] - z[0])  # °C/m

    if dT_dz > 0:
        # Extrapolate linearly: T(z) = T[0] + dT_dz*(z - z[0]) = 0
        z_temperate = z[0] - T[0] / dT_dz
        note = "linear extrap."
    else:
        z_temperate = float("inf")
        note = "gradient negative — temperatures decrease with depth"

    print(f"{site:<8} {dT_dz:+.4f} °C/m    {z_temperate:<28.1f} {note}")

print()
print("Notes:")
print("  Gradient > 0 means ice warms with depth (toward temperate).")
print("  Extrapolation assumes the observed near-surface gradient holds at depth,")
print("  which is a strong assumption — actual CTS depth is likely shallower due")
print("  to non-linear temperature profiles.")
